# Assessment 2: Refreshable Multi-Source Cyber Threat Intelligence (CTI) RAG Assistant

**Base model retained:** Google Gemma 2 2B Instruct (4-bit NF4)  
**Retrieval:** ChromaDB + BM25 hybrid retrieval  
**Embedding model:** `sentence-transformers/all-MiniLM-L6-v2`  
**Application:** CTI research and preliminary phishing-email triage

## 1. Introduction & Problem Statement
Cybersecurity analysts need to investigate both historical threat behaviour and more recent attacks, vulnerabilities and phishing infrastructure. Information is spread across incident summaries, CVE records, ransomware listings, CTI guidance and historical reports. Manually correlating these sources can slow triage and incident response.

**Target users:** SOC analysts, threat hunters, incident responders and cybersecurity researchers.

**Goal:** Build a multi-source Retrieval-Augmented Generation (RAG) assistant using the existing open-weights Gemma 2 2B Instruct model. The application retrieves relevant evidence, provides source/date context and generates concise answers for CTI research and preliminary phishing-email triage.

## Scope of This Project

This project follows Pathway A: Retrieval-Augmented Generation (RAG). It keeps the existing Gemma 2 2B model and improves the evidence collection and retrieval design.

1. **Problem:** Analysts need to find and correlate information across historical reports, incident summaries, vulnerability records and phishing indicators.
2. **Solution:** Use Chroma/BM25 retrieval for narrative CTI knowledge, structured lookups for CVE and ransomware records, and exact blocklist matching for email URLs.
3. **Boundary:** This is analyst decision support, not a malware sandbox, mail gateway or comprehensive live threat-intelligence platform. A blocklist match is a lead; a non-match does not establish safety. A human analyst must verify important findings.

## 2. Data sources and freshness strategy

This notebook uses a **multi-source RAG design**. It does not assume that one dataset contains every type of CTI evidence.

| Source | Purpose | Refresh / limitation |
|---|---|---|
| `threatcluster/threat-incident-clusters` | Recent incident summaries, entities, reporting dates and source outlets | Hugging Face snapshot generated 2026-09-06; periodic snapshot, not live. Summaries are model-generated and need corroboration. |
| `threatcluster/cve-exploitation-signals` | CVE descriptions, CVSS, affected products, CISA KEV status, exploit availability and ransomware-use field | Snapshot generated 2026-09-06; structured lookup, not a live vulnerability feed. |
| `threatcluster/ransomware-leak-site-victims` | Ransomware group claims, named organisations, sectors, countries and dates | Snapshot generated 2026-09-06. A leak-site listing is a criminal group's claim, not proof of a confirmed breach. |
| `reloading0101/threat-intelligence-dataset` | Curated CTI Q&A covering vulnerabilities, ATT&CK, email threats and incident response | Curated instruction/Q&A data grounded in public sources; no continuous update schedule is promised. |
| `phishdestroy/destroylist` | Exact domain/URL reputation lookup for links pasted into emails | Dataset card says it syncs hourly from GitHub. The application reads the feed timestamp and warns if it appears stale. It is a domain blocklist, not an email corpus. |
| ThreatCluster REST API | Live incident/entity/dark-web search at query time | Separate optional API source. Requires `THREATCLUSTER_API_KEY`; current free tier documentation states 100 credits/day and a seven-day lookback. API results are labelled [L#], not treated as snapshot datasets. |
| `guychuk/cyber-threat-intelligence-reports` | Historical CTI reports, including older campaigns such as MiniDuke | Retained as an archive, not treated as current threat intelligence. Verify its dataset provenance and rights before redistribution. |

**Important freshness distinction:** Hugging Face and Kaggle datasets are usually published snapshots. A frequently refreshed feed can be downloaded again, but a static snapshot does not become live simply because it is hosted on Hugging Face. This notebook refreshes the PhishDestroy file when the notebook starts; the ThreatCluster snapshots remain dated snapshots until their publisher releases a new revision. When enabled and configured, the ThreatCluster API adds a separately cited live search result at question time. It does not convert the stored snapshots into live feeds.

**Assessment honesty:** This system reports the dates and limitations available in the source data. It must not invent financial losses, claim an unverified ransomware listing is a confirmed breach, or describe a feed as current when its timestamp is stale.

# Design Decisions and Rationale

## 1. Why these components?

- **Embedding model — `sentence-transformers/all-MiniLM-L6-v2`:** A compact sentence-transformer that can create semantic representations efficiently in a Colab environment. It is a reasonable baseline for semantic retrieval, but its retrieval quality on CTI terminology must be measured rather than assumed.
- **Vector store — ChromaDB:** Stores chunk embeddings and supports similarity search. The notebook uses Chroma semantic retrieval plus BM25 keyword retrieval and fuses the ranked results. It returns up to 3 chunks (`top_k=3`); this value must be evaluated, not assumed optimal.
- **Prompt template:** Constrains the model to answer from retrieved CTI evidence, disclose when evidence is insufficient, and cite retrieved source IDs.

## 2. Why use RAG?

RAG is a suitable approach because the application must find specific facts in a document collection at query time. The corpus can be updated without retraining the base model. RAG is not the only possible solution, but it is a good fit for this document-question-answering use case.

## 3. Decoding and factuality

For the revised generation configuration, `do_sample=False` uses greedy decoding, reducing variation between repeated runs. Temperature is not used when sampling is disabled. Deterministic decoding does **not** guarantee factual answers; source-grounding, retrieval quality, and evaluation are still required.

## 4. Target users

- SOC analysts who need to search historical and periodically refreshed CTI records.
- Threat hunters and incident responders researching reported threat actors, campaigns, tools, and techniques.
- Cybersecurity students and researchers studying historical threat intelligence.

**Important limitation:** The application supports research and triage by retrieving historical report evidence. It does not independently detect malware, detonate files, validate live indicators, or replace analyst judgement.

## Scope boundary and output requirements

The application keeps the existing Gemma 2 2B model and RAG approach. It adds newer, task-specific sources rather than changing the language model.

The assistant can:
- Answer questions from historical reports and curated CTI records.
- Retrieve incident summaries and report dates.
- Look up CVEs, severity, affected products, KEV status and known exploit fields.
- Look up ransomware leak-site claims by group, organisation, sector or country.
- Check URLs/domains in a pasted email against the downloaded PhishDestroy blocklist.
- Explain observed email warning signs and provide safe analyst checks.

The assistant cannot prove that an email is malicious from text alone, execute or safely detonate files, establish a confirmed breach from a ransomware group's claim, or calculate a calibrated probability from vector similarity. If financial loss is not present in a source, it must say **not reported in the retrieved data**.

For a breach/incident question, the answer should distinguish: (1) what happened / the issue, (2) target and likely objective, (3) affected systems or data, (4) reported impact or loss, and (5) documented or recommended mitigation. Unknown fields must remain unknown rather than being guessed.

## Strict approved-source grounding rule
Every CTI claim, incident detail, loss figure, threat actor attribution, and mitigation action must be supported by a retrieved record from the selected datasets or the separately labelled live ThreatCluster API channel. The assistant must not fill gaps from Gemma's pretrained knowledge. If approved sources do not contain a fact, it must say: **"Not found in the selected datasets or live API results."** This applies field-by-field, not only when retrieval returns zero documents. The user's submitted email may be described as direct input evidence, but it must not be presented as a verified CTI finding.


## Question-intent routing and traceable citations

The CTI interface now distinguishes definition questions, mitigation questions, direct factual questions, and explicit case-study requests. A definition or mitigation question (for example, “What is a wiper attack and how can it be mitigated?”) must not be reformatted as two incident case studies. Case-study retrieval is used only when the question explicitly requests cases/examples or asks a broad sector-plus-attack question.

Generated factual statements are instructed to cite evidence labels such as `[S1]`, `[C1]`, `[V1]`, `[R1]`, and `[L1]`. The application appends dataset cards for snapshot evidence and the API reference/fetch time for live evidence. The evidence panel shows record title, source type, date, record ID, excerpt, and available links. If approved records do not support a requested detail, the answer must say **Not found in the selected datasets or live API results.**


**Live API configuration:** Add `THREATCLUSTER_API_KEY` to Colab Secrets (or an environment variable) to enable optional live ThreatCluster search. The UI toggle controls whether a query uses it. For email triage, the default is off; when enabled, only an extracted domain is sent, never the email text.


# Import packages

##1. Environment Setup (Code Cell)
Install the required libraries for Hugging Face, LangChain, and vector embeddings to build your implementation evidence.

In [ ]:
# Install compatible dependencies for Colab.
# If core packages are changed, restart the runtime before continuing.
%pip install -q bitsandbytes chromadb "gradio>=6,<7" datasets sentence-transformers transformers accelerate langchain langchain-community langchain-core langchain-huggingface langchain-text-splitters rank-bm25 fastapi uvicorn requests pandas numpy "opentelemetry-api>=1.39,<=1.42.1" "opentelemetry-sdk>=1.39,<=1.42.1"

## 1. Load the datasets

The notebook downloads public Hugging Face datasets. Several datasets are snapshots and may take time to download. The Gemma model may require accepting its access terms and providing a Hugging Face token; that authentication remains in the model-loading cell below.

## Data architecture note

Narrative sources (incident summaries, curated CTI knowledge and historical reports) are embedded for Chroma/BM25 retrieval. CVE and ransomware victim tables are queried as structured records only when relevant, so the notebook avoids embedding every row of the 60K-CVE and 20K-victim tables. The phishing-domain feed is also checked by exact domain matching rather than semantic similarity.

This separation improves traceability and reduces unnecessary embedding work. It does not guarantee a 15-second response: the local Gemma generation time is usually the dominant per-query cost. Check the retrieval and generation timings shown by the app.

In [ ]:
from datasets import load_dataset
import pandas as pd
import json

# Current/periodic Hugging Face CTI snapshots.
print("Loading ThreatCluster incident summaries...")
incident_ds = load_dataset("threatcluster/threat-incident-clusters", split="train")

print("Loading CVE exploitation records...")
cve_ds = load_dataset("threatcluster/cve-exploitation-signals", split="train")
cve_df = cve_ds.to_pandas()

print("Loading ransomware leak-site victim claims...")
ransomware_ds = load_dataset("threatcluster/ransomware-leak-site-victims", split="train")
ransomware_df = ransomware_ds.to_pandas()

print("Loading curated CTI Q&A knowledge...")
curated_cti_ds = load_dataset("reloading0101/threat-intelligence-dataset", split="train")

# Keep the original report collection as a historical archive so older campaigns
# (for example MiniDuke) remain searchable. It is NOT treated as live intelligence.
print("Loading historical CTI report archive...")
historical_ds = load_dataset("guychuk/cyber-threat-intelligence-reports", split="train")

print("\nDataset sizes")
print("Incident summaries:", len(incident_ds))
print("CVE records:", len(cve_df))
print("Ransomware listing claims:", len(ransomware_df))
print("Curated CTI Q&A:", len(curated_cti_ds))
print("Historical reports:", len(historical_ds))

# Build searchable text once for fast, structured lookups in the app.
import numpy as np

def _as_search_text(value):
    """Convert scalar, NumPy array, list and mapping values safely to text."""
    if value is None:
        return ""
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (list, tuple, dict, set)):
        if isinstance(value, set):
            value = sorted(value, key=str)
        return json.dumps(value, ensure_ascii=False, default=str)
    if pd.api.types.is_scalar(value) and pd.isna(value):
        return ""
    return str(value)

cve_search_columns = [c for c in ["cve_id", "description", "affected_vendors", "affected_products", "cwe_ids"] if c in cve_df.columns]
ransomware_search_columns = [c for c in ["group_name", "victim_name", "country", "sector"] if c in ransomware_df.columns]

cve_df["_search_blob"] = cve_df[cve_search_columns].apply(
    lambda row: " ".join(_as_search_text(v) for v in row.values).lower(), axis=1
)
ransomware_df["_search_blob"] = ransomware_df[ransomware_search_columns].apply(
    lambda row: " ".join(_as_search_text(v) for v in row.values).lower(), axis=1
)

print("Structured CVE and ransomware lookup indexes are ready.")

# Data Chunking (Text Splitting)

Narrative incident summaries, curated CTI knowledge and historical reports are split into overlapping chunks. CVE and ransomware-victim records are kept in structured tables and queried directly when the question is relevant. This avoids embedding every row of those large tables.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import requests
import json
import re
from datetime import datetime, timezone

# Convert any dataset field to readable, bounded text.
def as_text(value, limit=1800):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (dict, list, tuple, set)):
        if isinstance(value, set):
            value = sorted(value, key=str)
        value = json.dumps(value, ensure_ascii=False, default=str)
    return str(value)[:limit]

# Build a compact, in-memory lookup table for named incident case studies.
# This is created once at startup; it is NOT rebuilt for each question.
incident_df = incident_ds.to_pandas()
incident_case_columns = [
    c for c in [
        "title", "summary", "entities", "sources", "first_reported",
        "last_reported", "url", "cluster_id"
    ] if c in incident_df.columns
]
if incident_case_columns:
    incident_df["_case_search_blob"] = incident_df[incident_case_columns].apply(
        lambda row: " ".join(as_text(value, 2500) for value in row.values).lower(),
        axis=1
    )
else:
    incident_df["_case_search_blob"] = ""
print(f"Incident case lookup prepared: {len(incident_df):,} records.")

base_documents = []

# 1) Recent incident-cluster summaries. Titles/summaries are generated by the
# publisher and must be corroborated through their listed outlets when possible.
for row in incident_ds:
    title = as_text(row.get("title"), 300)
    summary = as_text(row.get("summary"), 1600)
    if not (title or summary):
        continue
    entities = as_text(row.get("entities"), 1200)
    outlets = as_text(row.get("sources"), 900)
    content = (
        f"Incident title: {title}\n"
        f"Incident summary: {summary}\n"
        f"Entities: {entities}\n"
        f"Reporting outlets: {outlets}\n"
        f"First reported: {as_text(row.get('first_reported'))}\n"
        f"Last reported: {as_text(row.get('last_reported'))}\n"
        f"Urgency: {as_text(row.get('urgency_level'))}\n"
        f"Threat score (publisher ranking, not a probability): {as_text(row.get('threat_score'))}\n"
        f"Severity score (publisher score): {as_text(row.get('severity_score'))}"
    )
    metadata = {
        "source_dataset": "threatcluster/threat-incident-clusters",
        "record_type": "incident_summary",
        "title": title or "Incident summary",
        "source_url": as_text(row.get("url"), 500),
        "record_date": as_text(row.get("last_reported") or row.get("first_reported"), 50),
        "record_id": as_text(row.get("cluster_id"), 100),
    }
    base_documents.append(Document(page_content=content, metadata=metadata))

# 2) Curated CTI instruction/answer records. These are secondary knowledge records,
# not primary news articles or original incident reports.
for row in curated_cti_ds:
    instruction = as_text(row.get("instruction"), 500)
    user_input = as_text(row.get("input"), 1000)
    output = as_text(row.get("output"), 2600)
    if not output:
        continue
    meta_obj = row.get("metadata") or {}
    if not isinstance(meta_obj, dict):
        meta_obj = {}
    category = as_text(meta_obj.get("category"), 100)
    grounding = as_text(meta_obj.get("grounding"), 250)
    source_refs = as_text(meta_obj.get("source_refs"), 700)
    content = (
        f"Curated CTI task: {instruction}\n"
        f"Input/entity: {user_input}\n"
        f"Category: {category}\n"
        f"Grounding sources: {grounding}\n"
        f"Source references: {source_refs}\n"
        f"Curated CTI answer: {output}"
    )
    metadata = {
        "source_dataset": "reloading0101/threat-intelligence-dataset",
        "record_type": "curated_cti_qa",
        "title": category or "Curated CTI knowledge",
        "source_url": "https://huggingface.co/datasets/reloading0101/threat-intelligence-dataset",
        "record_id": as_text(meta_obj.get("id"), 100),
    }
    base_documents.append(Document(page_content=content, metadata=metadata))

# 3) Historical archive. Retained for older campaigns and historical questions.
for row in historical_ds:
    report_text = as_text(row.get("text"), 100000)
    if not report_text.strip():
        continue
    report_url_match = re.search(r"Report_URL\s*:\s*(https?://\S+)", report_text)
    metadata = {
        "source_dataset": "guychuk/cyber-threat-intelligence-reports",
        "record_type": "historical_report",
        "title": as_text(row.get("title"), 300) or "Historical CTI report",
        "path": as_text(row.get("path"), 300),
        "record_id": as_text(row.get("id"), 100),
        "start_line": as_text(row.get("start_line"), 30),
        "end_line": as_text(row.get("end_line"), 30),
    }
    if report_url_match:
        metadata["source_url"] = report_url_match.group(1).rstrip(".,)")
    base_documents.append(Document(page_content=report_text, metadata=metadata))

# 4) PhishDestroy feed. This is used for exact URL/domain IOC matching, NOT vector search.
# The dataset card says it syncs hourly; check the published timestamp instead of assuming.
IOC_DOMAINS = set()
IOC_FEED_LAST_UPDATED = "Timestamp unavailable"
IOC_FEED_FETCHED_AT = datetime.now(timezone.utc).isoformat()
IOC_FEED_STATUS = "Unavailable"

IOC_FEED_URL = "https://huggingface.co/datasets/phishdestroy/destroylist/resolve/main/domains.txt"
IOC_README_URL = "https://huggingface.co/datasets/phishdestroy/destroylist/resolve/main/README.md?download=true"

try:
    feed_response = requests.get(IOC_FEED_URL, timeout=25, headers={"User-Agent": "CTI-RAG-Student-Project/1.0"})
    feed_response.raise_for_status()
    for line in feed_response.text.splitlines():
        candidate = line.strip().lower()
        if not candidate or candidate.startswith("#"):
            continue
        candidate = re.sub(r"^https?://", "", candidate)
        candidate = candidate.split("/")[0].split(":")[0].strip(".")
        if candidate:
            IOC_DOMAINS.add(candidate)

    readme_response = requests.get(IOC_README_URL, timeout=15, headers={"User-Agent": "CTI-RAG-Student-Project/1.0"})
    if readme_response.ok:
        timestamp_match = re.search(r"Last updated:\s*([^\r\n]+)", readme_response.text, re.I)
        if timestamp_match:
            IOC_FEED_LAST_UPDATED = timestamp_match.group(1).strip()
    IOC_FEED_STATUS = "Loaded"
except Exception as feed_error:
    print("WARNING: PhishDestroy feed could not be downloaded:", repr(feed_error))
    print("Email URL reputation will be shown as NOT CHECKED; it will not be treated as safe.")

print(f"PhishDestroy feed status: {IOC_FEED_STATUS}; domains loaded: {len(IOC_DOMAINS):,}")
print("PhishDestroy published timestamp:", IOC_FEED_LAST_UPDATED)

# This feed claims hourly refreshes; flag a published timestamp older than 6 hours.
try:
    parsed_feed_time = datetime.strptime(IOC_FEED_LAST_UPDATED.replace(" UTC", ""), "%Y-%m-%d %H:%M").replace(tzinfo=timezone.utc)
    feed_age_hours = (datetime.now(timezone.utc) - parsed_feed_time).total_seconds() / 3600
    if feed_age_hours > 6:
        print(f"WARNING: The published IOC snapshot is {feed_age_hours:.1f} hours old. Treat matches as a lead and verify independently.")
except Exception:
    print("WARNING: Could not parse the IOC feed timestamp. Verify feed freshness before relying on it.")

# 5) CVE and ransomware target rows remain structured tables; we look them up only
# when a query is about a vulnerability or ransomware targeting. This avoids embedding
# tens of thousands of rows unnecessarily.
print(f"\nBuilding vector-search collection from {len(base_documents):,} narrative/curated records...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=120,
    add_start_index=True,
    separators=["\n\n", "\n", ". ", " ", ""]
)
docs = text_splitter.split_documents(base_documents)

print(f"Loaded {len(base_documents):,} base records and created {len(docs):,} chunks.")
print("Example metadata:", docs[0].metadata if docs else "No chunks created.")

# Create the Vector Database

The narrative CTI chunks are embedded with the same sentence-transformer model and stored in ChromaDB. At query time, Chroma semantic similarity is combined with BM25 keyword search. Exact CVE and ransomware-victim lookups are handled separately.

**Strict grounding requirement:** A retrieved record must directly match the question. The app uses a conservative evidence gate before generation, and Gemma is instructed to use only supplied selected-dataset records and separately labelled live API results. Missing details are labelled "Not found in the selected datasets or live API results."

### Retrieval and evidence-gate update
The application distinguishes a request for a named case study from a normal technical explanation or comparison. A missing row in the incident-cluster table does not automatically block generation if a relevant historical report or curated CTI record was retrieved. Every generated factual statement must still be supported by those retrieved records. If no relevant evidence is retrieved, the assistant abstains. The hybrid search retrieves a wider candidate set and expands common C2, encryption, obfuscation, MiniDuke and zero-day terminology before ranking.


In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from rank_bm25 import BM25Okapi
import re
import numpy as np

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Generating embeddings and building the vector store...")
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="cti_refreshable_multisource"
)

# Keyword index complements semantic retrieval for exact CVE IDs, malware names and techniques.
def tokenize_cti(text):
    return re.findall(r"(?u)[a-z0-9]+(?:[._-][a-z0-9]+)*", str(text).lower())

print("Building BM25 keyword index...")
bm25_corpus = [tokenize_cti(doc.page_content) for doc in docs]
bm25_index = BM25Okapi(bm25_corpus)

# Precompute title matches once. Do not scan every document on each MiniDuke query.
MINIDUKE_TITLE_INDICES = [
    i for i, doc in enumerate(docs)
    if "miniduke" in str((doc.metadata or {}).get("title", "")).lower()
]

TOP_K = 5
CANDIDATE_K = 18


def is_reference_noise(doc):
    text = doc.page_content or ""
    low = text.lower()
    first_lines = "\n".join(text.splitlines()[:6]).lower()
    if "appendix: parallel and previous research" in low:
        return True
    if re.search(r"(?im)^\s*(references|bibliography)\s*$", first_lines):
        return True
    url_count = len(re.findall(r"https?://", text, flags=re.I))
    year_entries = len(re.findall(r"(?m)^\s*(?:\[\d+\]|\d+[.)])?.{0,140}\b(?:19|20)\d{2}\b", text))
    return url_count >= 3 and year_entries >= 2


def expand_cti_query(query):
    """Expand CTI terminology without changing the analyst's original question."""
    q = str(query or "").strip()
    low = q.lower()
    extras = []

    if "miniduke" in low or "mini duke" in low:
        extras.extend([
            "MiniDuke PDF exploit Adobe Reader CVE-2013-0640",
            "MiniDuke downloader assembler backdoor malware",
            "MiniDuke command and control Twitter Google Search encrypted GIF"
        ])

    if any(term in low for term in ("command-and-control", "command and control", "c2", "c&c")):
        extras.extend([
            "command and control C2 communications",
            "MITRE ATT&CK T1071 application layer protocol",
            "T1573 encrypted channel",
            "T1090 proxy",
            "beaconing callback traffic"
        ])

    if any(term in low for term in ("obfuscat", "encryption", "encrypted", "cipher")):
        extras.extend([
            "MITRE ATT&CK T1027 obfuscated files or information",
            "MITRE ATT&CK T1573 encrypted channel",
            "TLS HTTPS DNS tunneling encoding packing",
            "AES RC4 RSA cryptographic protocol"
        ])

    if "zero-day" in low or "zero day" in low or "0-day" in low:
        extras.extend(["0-day", "zero-day exploit", "CVE vulnerability", "unpatched vulnerability"])

    # Keep expansions bounded to avoid drowning out the analyst's actual terms.
    return (q + " " + " ".join(dict.fromkeys(extras)))[:1800]


def _doc_key(doc):
    meta = doc.metadata or {}
    return (
        str(meta.get("source_dataset", "")),
        str(meta.get("record_id", meta.get("id", ""))),
        str(meta.get("start_index", "")),
        (doc.page_content or "")[:100]
    )


def hybrid_retrieve(query, k=TOP_K, candidate_k=CANDIDATE_K):
    search_query = expand_cti_query(query)
    vector_docs = vectorstore.similarity_search(search_query, k=candidate_k)
    # BM25 only needs a compact, de-duplicated token list; full email bodies are
    # reduced to a short retrieval query before this function is called.
    bm25_stopwords = {
        "what", "which", "when", "where", "who", "why", "how", "can", "could",
        "would", "should", "please", "explain", "compare", "describe", "give",
        "show", "list", "the", "and", "for", "from", "with", "about", "this",
        "that", "into", "their", "there", "are", "was", "were", "is", "are",
        "does", "did", "not", "use", "used", "using", "similar", "example",
        "examples", "step", "steps", "detail", "detailed"
    }
    query_tokens = [
        token for token in dict.fromkeys(tokenize_cti(search_query))
        if token not in bm25_stopwords and len(token) > 1
    ][:28]
    bm25_scores = bm25_index.get_scores(query_tokens)

    # argpartition avoids sorting the entire corpus just to obtain a few candidates.
    if len(bm25_scores) > candidate_k:
        candidate_indices = np.argpartition(bm25_scores, -candidate_k)[-candidate_k:]
        lexical_indices = candidate_indices[np.argsort(bm25_scores[candidate_indices])[::-1]]
    else:
        lexical_indices = np.argsort(bm25_scores)[::-1]
    lexical_docs = [docs[int(i)] for i in lexical_indices if float(bm25_scores[int(i)]) > 0]

    # Reuse the precomputed MiniDuke title index rather than scanning all chunks.
    title_docs = []
    if "miniduke" in str(query).lower() and MINIDUKE_TITLE_INDICES:
        titled = sorted(
            MINIDUKE_TITLE_INDICES,
            key=lambda i: float(bm25_scores[i]),
            reverse=True
        )
        title_docs = [docs[i] for i in titled[:candidate_k] if float(bm25_scores[i]) > 0]

    fused = {}
    for ranked_docs, weight in [(vector_docs, 1.0), (lexical_docs, 1.35), (title_docs, 1.5)]:
        for rank, doc in enumerate(ranked_docs, start=1):
            if is_reference_noise(doc):
                continue
            key = _doc_key(doc)
            if key not in fused:
                fused[key] = {"doc": doc, "score": 0.0}
            fused[key]["score"] += weight / (60 + rank)

    ordered = sorted(fused.values(), key=lambda item: item["score"], reverse=True)
    return [item["doc"] for item in ordered[:k]]

print(f"Hybrid retrieval ready: Chroma + BM25; top_k={TOP_K}.")

# Authenticate and Load the LLM (Gemma 2 2B)

To use Gemma, must pass the Hugging Face token.

In [ ]:
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)
from langchain_huggingface import HuggingFacePipeline

# Gemma access may require accepting the model terms on Hugging Face.
hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
if not hf_token:
    raise ValueError(
        "HF_TOKEN is missing. In Colab, add HF_TOKEN to Colab Secrets. "
        "In Hugging Face Spaces, add HF_TOKEN under Space Settings > Variables and secrets."
    )

os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

model_id = "google/gemma-2-2b-it"

if not torch.cuda.is_available():
    raise RuntimeError(
        "This 4-bit bitsandbytes configuration requires a compatible CUDA GPU. "
        "In Colab, select Runtime > Change runtime type > GPU."
    )

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=hf_token,
    device_map="auto",
    quantization_config=quantization_config
)
model.eval()
model.config.use_cache = True
if getattr(model, "generation_config", None) is not None:
    model.generation_config.use_cache = True

# A modest cap supports up to two short case studies plus mitigation.
# Keep the response concise; increase only if your tested outputs are truncated.
MAX_NEW_TOKENS = 210

generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,       # Greedy decoding.
    num_beams=1,           # No beam-search overhead.
    use_cache=True,        # Reuse attention key/value states while generating.
    return_full_text=False,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
)

llm = HuggingFacePipeline(pipeline=generation_pipeline)
print("Gemma 2 2B Instruct loaded successfully in 4-bit mode.")
print("GPU:", torch.cuda.get_device_name(0))
print(f"Generation cap: {MAX_NEW_TOKENS} new tokens.")

# Warm up CUDA and the model once during startup, not during the first user query.
print("Warming up the model...")
with torch.inference_mode():
    _ = generation_pipeline(
        "<start_of_turn>user\nReply OK.<end_of_turn>\n<start_of_turn>model\n",
        max_new_tokens=2
    )
torch.cuda.synchronize()
print("Model warm-up complete.")

# Construct the RAG Prompt Template

A highly disciplined prompt is necessary to control hallucination and ensure the model only uses the retrieved chunks.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import re
import pandas as pd
from urllib.parse import urlparse


DATASET_CATALOG = {
    "threatcluster/threat-incident-clusters": {"owner": "ThreatCluster", "year": "2026", "title": "threat-incident-clusters", "url": "https://huggingface.co/datasets/threatcluster/threat-incident-clusters", "note": "Snapshot generated 2026-09-06; summaries are publisher-generated clusters.", "license": "CC BY 4.0"},
    "threatcluster/cve-exploitation-signals": {"owner": "ThreatCluster", "year": "2026", "title": "cve-exploitation-signals", "url": "https://huggingface.co/datasets/threatcluster/cve-exploitation-signals", "note": "Snapshot generated 2026-09-06.", "license": "CC BY 4.0"},
    "threatcluster/ransomware-leak-site-victims": {"owner": "ThreatCluster", "year": "2026", "title": "ransomware-leak-site-victims", "url": "https://huggingface.co/datasets/threatcluster/ransomware-leak-site-victims", "note": "Snapshot generated 2026-09-06; listings are group claims, not independently verified breaches.", "license": "CC BY 4.0"},
    "reloading0101/threat-intelligence-dataset": {"owner": "reloading0101", "year": "n.d.", "title": "threat-intelligence-dataset", "url": "https://huggingface.co/datasets/reloading0101/threat-intelligence-dataset", "note": "Curated CTI question/answer records; secondary knowledge source.", "license": "CC BY-NC-SA 4.0"},
    "guychuk/cyber-threat-intelligence-reports": {"owner": "guychuk", "year": "n.d.", "title": "cyber-threat-intelligence-reports", "url": "https://huggingface.co/datasets/guychuk/cyber-threat-intelligence-reports", "note": "Historical report archive; not a live feed.", "license": "See dataset card"},
    "phishdestroy/destroylist": {"owner": "phishdestroy", "year": "n.d.", "title": "destroylist", "url": "https://huggingface.co/datasets/phishdestroy/destroylist", "note": "Phishing/scam domain feed; freshness must be checked using its published timestamp.", "license": "See dataset card"}
}


def _dataset_info(dataset_name):
    return DATASET_CATALOG.get(dataset_name, {
        "owner": "Unknown publisher", "year": "n.d.",
        "title": dataset_name or "Unknown dataset", "url": "", "note": ""
    })


def format_docs(retrieved_docs, max_chars_per_doc=700, label_prefix="S"):
    """Format evidence with stable labels and clear dataset provenance."""
    formatted = []
    for index, doc in enumerate(retrieved_docs, start=1):
        meta = doc.metadata or {}
        dataset_name = meta.get("source_dataset", "Unknown dataset")
        info = _dataset_info(dataset_name)
        label = f"{label_prefix}{index}"
        title = meta.get("title", "Untitled CTI record")
        record_type = meta.get("record_type", "CTI record")
        date = meta.get("record_date", "Date not supplied")
        record_id = meta.get("record_id", "Not supplied")
        source_url = meta.get("source_url", "")
        if source_url == info.get("url"):
            source_url = ""  # Do not mislabel a dataset-card URL as an original incident report.
        evidence = (doc.page_content or "")[:max_chars_per_doc]
        formatted.append(
            f"[{label}]\n"
            f"Title: {title}\n"
            f"Record type: {record_type}\n"
            f"Dataset: {dataset_name}\n"
            f"Dataset card: {info['url'] or 'Not supplied'}\n"
            f"Record date: {date}\n"
            f"Record ID: {record_id}\n"
            f"Original record/source URL: {source_url or 'Not supplied'}\n"
            f"Evidence: {evidence}"
        )
    return "\n\n".join(formatted)


_STOP_WORDS = {
    "what", "which", "when", "where", "from", "with", "about", "does", "show",
    "reports", "report", "data", "dataset", "attack", "attacks", "incident", "incidents",
    "security", "cyber", "threat", "target", "targets", "impact", "loss", "losses",
    "explain", "please", "tell", "give", "latest", "current", "recent", "find", "known"
}


def _query_terms(query, max_terms=8):
    terms = re.findall(r"[a-z0-9][a-z0-9._-]{2,}", str(query).lower())
    return [t for t in dict.fromkeys(terms) if t not in _STOP_WORDS][:max_terms]


def _rank_rows(frame, terms, max_rows=3):
    """Rank structured records with one vectorised text scan per query."""
    if frame.empty or not terms:
        return frame.iloc[0:0]

    clean_terms = list(dict.fromkeys(str(t).lower() for t in terms if len(str(t)) >= 3))
    if not clean_terms:
        return frame.iloc[0:0]

    # One combined regex is substantially cheaper than scanning the 60k-row
    # CVE frame once for every query term.
    alternatives = "|".join(re.escape(t) for t in sorted(clean_terms, key=len, reverse=True))
    pattern = rf"(?<![a-z0-9])(?:{alternatives})(?![a-z0-9])"
    matches = frame["_search_blob"].str.findall(pattern, flags=re.I)
    scores = matches.str.len().fillna(0).astype("int32")
    matched = frame.loc[scores > 0].copy()
    if matched.empty:
        return matched

    matched["_match_score"] = scores.loc[matched.index]
    sort_cols = ["_match_score"]
    ascending = [False]
    if "published_date" in matched.columns:
        sort_cols.append("published_date")
        ascending.append(False)
    elif "discovered" in matched.columns:
        sort_cols.append("discovered")
        ascending.append(False)
    return matched.sort_values(sort_cols, ascending=ascending).head(max_rows)


# Topic-aware case-study lookup: for broad sector/attack questions, select up to
# two named incidents from the incident-summary dataset instead of returning only
# ransomware-group names. These records are summaries, so the linked reporting
# outlets should be checked before treating generated details as independently verified.
_CASE_SECTOR_PATTERNS = {
    "healthcare": [r"\bhealth[\s-]?care\b", r"\bhospitals?\b", r"\bclinics?\b", r"\bmedical (?:cent(?:er|re)|provider|system)\b", r"\bpatient care\b"],
    "education": [r"\bschools?\b", r"\buniversit(?:y|ies)\b", r"\beducation\b"],
    "finance": [r"\bbanks?\b", r"\bfinancial (?:services|institutions?)\b", r"\bpayment (?:processor|systems?)\b"],
    "government": [r"\bgovernment\b", r"\bmunicipalit(?:y|ies)\b", r"\bcity (?:council|government)\b", r"\bpublic sector\b"],
}
_CASE_ATTACK_PATTERNS = {
    "ransomware": [r"\bransomware\b", r"\bdouble extortion\b", r"\bdata extortion\b"],
    "phishing": [r"\bphishing\b", r"\bspear.?phishing\b", r"\bcredential theft\b"],
    "apt": [r"\bapt\b", r"\badvanced persistent threat\b", r"\bstate.?sponsored\b", r"\bcyber espionage\b"],
    "data breach": [r"\bdata breach\b", r"\bdata theft\b", r"\bexfiltrat(?:ion|ed)\b"],
}


def retrieve_incident_case_studies(query, max_cases=2):
    """Return up to two distinct named incident records relevant to broad sector/attack questions."""
    low = str(query).lower()
    active_sector_groups = [
        (name, pats) for name, pats in _CASE_SECTOR_PATTERNS.items()
        if any(re.search(pat, low, re.I) for pat in pats)
    ]
    active_attack_groups = [
        (name, pats) for name, pats in _CASE_ATTACK_PATTERNS.items()
        if any(re.search(pat, low, re.I) for pat in pats)
    ]
    # Keep this feature focused on requests that specify both a target sector and an attack type.
    if not active_sector_groups or not active_attack_groups or incident_df.empty:
        return []

    sector_patterns = [pat for _, pats in active_sector_groups for pat in pats]
    attack_patterns = [pat for _, pats in active_attack_groups for pat in pats]
    sector_regex = "(?:" + "|".join(sector_patterns) + ")"
    attack_regex = "(?:" + "|".join(attack_patterns) + ")"

    blobs = incident_df["_case_search_blob"].fillna("")
    sector_match = blobs.str.contains(sector_regex, regex=True, na=False)
    attack_match = blobs.str.contains(attack_regex, regex=True, na=False)
    candidates = incident_df.loc[sector_match & attack_match].copy()
    if candidates.empty:
        return []

    # Prefer records with more reporting outlets and higher publisher severity.
    # These are selection aids, not proof that one incident was more damaging.
    if "source_count" in candidates.columns:
        candidates["_source_rank"] = pd.to_numeric(candidates["source_count"], errors="coerce").fillna(0)
    else:
        candidates["_source_rank"] = 0
    if "severity_score" in candidates.columns:
        candidates["_severity_rank"] = pd.to_numeric(candidates["severity_score"], errors="coerce").fillna(0)
    else:
        candidates["_severity_rank"] = 0
    sort_cols = ["_source_rank", "_severity_rank"]
    ascending = [False, False]
    if "last_reported" in candidates.columns:
        sort_cols.append("last_reported")
        ascending.append(False)
    candidates = candidates.sort_values(sort_cols, ascending=ascending, kind="stable")

    case_docs = []
    seen = set()
    for _, row in candidates.iterrows():
        title = as_text(row.get("title"), 350) or "Untitled incident"
        cluster_id = as_text(row.get("cluster_id"), 100)
        dedupe_key = cluster_id or title.lower()
        if dedupe_key in seen:
            continue
        seen.add(dedupe_key)
        summary = as_text(row.get("summary"), 2200)
        entities = as_text(row.get("entities"), 1000)
        outlets = as_text(row.get("sources"), 900)
        first_reported = as_text(row.get("first_reported"), 50)
        last_reported = as_text(row.get("last_reported"), 50)
        source_url = as_text(row.get("url"), 600)
        content = (
            f"Named incident case: {title}\n"
            f"Summary: {summary}\n"
            f"Entities extracted by dataset publisher: {entities}\n"
            f"Reporting outlets: {outlets}\n"
            f"First reported: {first_reported}\n"
            f"Last reported: {last_reported}\n"
            "Evidence caveat: This is a generated incident-cluster summary. Verify key facts, impact and response in the linked reporting outlets. "
            "Do not infer financial loss or mitigation if not stated."
        )
        metadata = {
            "source_dataset": "threatcluster/threat-incident-clusters",
            "record_type": "incident_case_study",
            "title": title,
            "source_url": source_url,
            "record_date": last_reported or first_reported,
            "record_id": cluster_id,
        }
        case_docs.append(Document(page_content=content, metadata=metadata))
        if len(case_docs) >= max_cases:
            break
    return case_docs


def retrieve_structured_context(query):
    """Look up CVE and ransomware victim rows only when relevant to the query."""
    q = str(query)
    low = q.lower()
    sections = []

    # CVE lookup: exact CVE IDs first; otherwise a small ranked product/vendor search.
    cve_ids = re.findall(r"CVE-\d{4}-\d{3,}", q, flags=re.I)
    cve_rows = cve_df.iloc[0:0]
    if cve_ids and "cve_id" in cve_df.columns:
        cve_rows = cve_df[cve_df["cve_id"].astype(str).str.upper().isin([x.upper() for x in cve_ids])].head(3)
    elif re.search(r"\b(cve|vulnerab\w*|exploit\w*|patch\w*|kev|affected products?)\b", low):
        cve_rows = _rank_rows(cve_df, _query_terms(q), max_rows=3)

    for i, (_, row) in enumerate(cve_rows.iterrows(), start=1):
        sections.append(
            f"[V{i}]\n"
            f"CVE: {as_text(row.get('cve_id'))}\n"
            f"Description: {as_text(row.get('description'), 900)}\n"
            f"Published date: {as_text(row.get('published_date'))}\n"
            f"CVSS v3: {as_text(row.get('cvss_v3_score'))} ({as_text(row.get('cvss_v3_severity'))})\n"
            f"Affected vendors: {as_text(row.get('affected_vendors'), 500)}\n"
            f"Affected products: {as_text(row.get('affected_products'), 500)}\n"
            f"CISA KEV listed in this snapshot: {as_text(row.get('in_kev'))}\n"
            f"KEV added / due dates: {as_text(row.get('kev_added_date'))} / {as_text(row.get('kev_due_date'))}\n"
            f"Public exploit known / count: {as_text(row.get('has_exploit'))} / {as_text(row.get('exploit_count'))}\n"
            f"Ransomware use field: {as_text(row.get('ransomware_use'))}\n"
            "Dataset: threatcluster/cve-exploitation-signals. "
            "Dataset card: https://huggingface.co/datasets/threatcluster/cve-exploitation-signals. "
            "Snapshot generated 2026-09-06."
        )

    # Victim records are only appropriate when the user asks about leak-site claims/listings.
    # Do not let group-name-only rows replace concrete incident case studies.
    if re.search(r"\b(leak[\s-]?site|victim (?:listings?|claims?)|ransomware listings?|which organisations were listed)\b", low):
        victim_rows = _rank_rows(ransomware_df, _query_terms(q), max_rows=3)
        for i, (_, row) in enumerate(victim_rows.iterrows(), start=1):
            sections.append(
                f"[R{i}]\n"
                f"Group claim: {as_text(row.get('group_name'))}\n"
                f"Named organisation: {as_text(row.get('victim_name'))}\n"
                f"Sector / country: {as_text(row.get('sector'))} / {as_text(row.get('country'))}\n"
                f"First observed / published: {as_text(row.get('discovered'))} / {as_text(row.get('published'))}\n"
                f"Delisted: {as_text(row.get('delisted'))}\n"
                "Dataset: threatcluster/ransomware-leak-site-victims. "
                "Dataset card: https://huggingface.co/datasets/threatcluster/ransomware-leak-site-victims. "
                "Important caveat: this is a ransomware-group leak-site claim, not independent proof of a confirmed breach, data loss or payment. "
                "The dataset does not contain a verified financial-loss amount. Snapshot generated 2026-09-06."
            )

    return "\n\n".join(sections) if sections else "No matching structured CVE or ransomware-target records were selected for this query."


def extract_email_urls(text):
    return re.findall(r"(?:https?://|www\.)[^\s<>\]\[()]+", text or "", flags=re.I)


def _normalise_domain(url):
    value = str(url).strip().rstrip(".,);]")
    if value.lower().startswith("www."):
        value = "http://" + value
    elif not re.match(r"^https?://", value, flags=re.I):
        value = "http://" + value
    try:
        host = (urlparse(value).hostname or "").lower().strip(".")
        return host[4:] if host.startswith("www.") else host
    except Exception:
        return ""


def check_email_domains(email_text):
    """Exact/suffix domain matching against the separately downloaded IOC set."""
    urls = extract_email_urls(email_text)
    results = []
    for url in urls:
        domain = _normalise_domain(url)
        if not domain:
            continue
        # Check the domain and each parent domain using set lookups (fast even for large feeds).
        matches = []
        candidate = domain
        while candidate:
            if candidate in IOC_DOMAINS:
                matches.append(candidate)
            if "." not in candidate:
                break
            candidate = candidate.split(".", 1)[1]
        results.append({"url": url, "domain": domain, "matches": matches[:5]})
    return results


# CTI Q&A: answer the exact question, but surface incident/target/impact/mitigation
# only when the evidence actually supports those fields.
gemma_prompt_template = """<start_of_turn>user
You are a strict, evidence-bound Cyber Threat Intelligence analyst. You are NOT a general-knowledge answer bot.

SOURCE RULES:
1. Use ONLY the supplied selected-dataset records and the separately labelled live ThreatCluster API records. Never use pretrained knowledge, memory, assumptions or unlisted websites to fill gaps.
2. Every factual statement must cite a supplied evidence label: [S1], [S2], [C1], [C2], [V1], [R1], or [L1]. Never invent a citation label.
3. Use a record only when it directly supports the point. Similarity does not prove relevance.
4. If a requested fact is absent from all supplied approved sources, write exactly: "Not found in the selected datasets or live API results." Do not guess.
5. Never invent an incident, actor, target, date, statistic, loss, impact, mitigation or citation. Do not merge different incidents.
6. A ransomware leak-site listing is a group's claim, not independent proof of a confirmed breach, data theft, payment or financial loss.
7. Incident-cluster summaries are publisher-generated summaries. Attribute claims to the relevant dataset or live API record. Verify key claims in linked original reporting where possible.
8. Mitigation must be explicitly documented in the supplied records. Do not add general advice from model memory. If absent, say it is not found.
9. Similarity scores and publisher threat/severity scores are not probabilities.
10. Live API results are a distinct source channel with their own timestamp, API endpoint and credit metadata. Do not describe them as dataset snapshots.
11. Do not output a generic statement that only selected datasets were used. The application appends source names and links after your answer.

QUESTION-SPECIFIC RESPONSE INSTRUCTION:
{response_instruction}

CASE STUDY RECORDS:
{case_study_context}

NARRATIVE RETRIEVED RECORDS:
{context}

STRUCTURED CVE / RANSOMWARE RECORDS:
{structured_context}

USER QUESTION:
{input}
<end_of_turn>
<start_of_turn>model
"""

prompt = PromptTemplate(
    template=gemma_prompt_template,
    input_variables=["context", "structured_context", "case_study_context", "input", "response_instruction"]
)
qa_chain = prompt | llm | StrOutputParser()


email_triage_template = """<start_of_turn>user
You are a strict, evidence-bound SOC email triage assistant.

SOURCE BOUNDARIES:
- You may describe observable features that are literally present in the submitted email. Label these as observations from the submitted email, not as facts from a CTI dataset.
- For historical campaigns, threat actors, malware families, known malicious domains, confirmed breaches, impacts, financial losses, and mitigation actions, use ONLY the supplied selected-dataset evidence, the separately labelled live API evidence, and the domain-feed result.
- Do NOT use pretrained/general knowledge to fill missing CTI facts. Do NOT invent a threat actor, campaign, malware, breach, victim, loss amount, or mitigation.
- If a requested CTI fact is absent from all approved sources, write exactly: "Not found in the selected datasets or live API results."
- A blocklist match is a lead, not definitive proof that the email is malicious. A non-match is NOT proof of safety. If the feed was unavailable or stale, say so.
- Do not open, execute, or detonate URLs, attachments, or scripts.
- Do not invent a risk percentage. Preserve the preliminary rule-based label exactly: {rule_assessment}.
- Actual loss is not established by the email itself; only state a loss when an approved source record explicitly documents it.
- Mitigation/response actions: include only actions explicitly stated in supplied approved-source records. If absent, say "Not found in the selected datasets or live API results." Do not add generic recommendations from model memory.
- Live API search for email triage must receive only an extracted domain, never the pasted email body or full email text.

Return concise plain language using this structure:
## Email triage result
**Assessment:** {rule_assessment}
**Reason:** [1-2 sentences based only on observed email text and dataset matches. Do not overstate certainty.]

## Observed warning signs
[2-4 bullets. Every point must be visible in the submitted email. If none, say so.]

## Possible objective and target
- **Possible objective:** [only if supported by email wording or approved source evidence; otherwise exact not-found wording]
- **Target:** [only what the email identifies or an approved source directly supports; otherwise exact not-found wording]
- **Actual loss:** [state only if explicitly documented in an approved record; otherwise exact not-found wording]

## Domain-feed check
[Report exact match/no match and feed timestamp/status. Explain the limits of the check.]

## Related CTI evidence
[Only directly relevant selected-dataset or live API evidence with source labels. If none, write: "Not found in the selected datasets or live API results."]

## Documented mitigation / response
[Only what approved source records explicitly say. If none, write: "Not found in the selected datasets or live API results."]

## Evidence limitations
[Briefly distinguish email observations from dataset-supported intelligence.]

Observed indicators:
{indicators}

Domain-feed results:
{ioc_results}

Feed published timestamp: {feed_timestamp}
Feed status: {feed_status}

Submitted email:
{email_text}

Retrieved selected-dataset and separately labelled live API CTI evidence:
{context}
<end_of_turn>
<start_of_turn>model
"""

email_triage_prompt = PromptTemplate(
    template=email_triage_template,
    input_variables=["indicators", "ioc_results", "feed_timestamp", "feed_status", "email_text", "context", "rule_assessment"]
)
email_triage_chain = email_triage_prompt | llm | StrOutputParser()


def detect_email_indicators(email_text: str):
    """Transparent text indicators; not a probability or final phishing verdict."""
    text = email_text or ""
    lower = text.lower()
    findings = []

    if re.search(r"\b(urgent|immediately|act now|final warning|within\s+\d+\s+hours?)\b|account.{0,40}(suspend|disable|lock|terminate|expire)", lower, re.I):
        findings.append("Pressure or account-threat language is present.")
    if re.search(r"(enter|provide|confirm|verify|submit|update|reset|share).{0,70}(password|passcode|credentials|one[- ]?time code|MFA code|login details)|(password|passcode|credentials|one[- ]?time code|MFA code|login details).{0,70}(enter|provide|confirm|verify|submit|update|reset|share)", lower, re.I | re.S):
        findings.append("The message appears to request credentials or authentication information.")
    urls = extract_email_urls(text)
    if urls:
        findings.append(f"The message contains {len(urls)} visible URL(s).")
    if re.search(r"\b(wire transfer|bank account|payment details|gift card|change.{0,25}bank details)\b", lower, re.I):
        findings.append("The message requests or discusses a payment or banking action.")
    if re.search(r"\.(exe|scr|js|jse|vbs|vbe|hta|iso|img|lnk|bat|cmd|ps1|docm|xlsm)\b", lower, re.I):
        findings.append("A potentially executable or macro-enabled attachment is mentioned.")
    if re.search(r"password[- ]protected|enable macros|enable content|disable security|protected document", lower, re.I):
        findings.append("The message asks the recipient to bypass a security control or enable document content.")
    if not findings:
        findings.append("No matches were found by the built-in text checklist; this does not establish that the email is safe.")
    return findings


def classify_email_preliminary(email_text: str, findings):
    findings_text = " ".join(findings).lower()
    if "no matches were found" in findings_text:
        return "INSUFFICIENT EVIDENCE"
    credential = "request credentials" in findings_text
    payment = "payment or banking action" in findings_text
    pressure = "pressure or account-threat" in findings_text
    urls = "visible url" in findings_text
    attachment = "attachment is mentioned" in findings_text or "bypass a security control" in findings_text
    if (credential or payment) and (pressure or urls or attachment):
        return "POTENTIALLY SUSPICIOUS"
    if len(findings) >= 2:
        return "NEEDS FURTHER REVIEW"
    return "NEEDS FURTHER REVIEW"

In [ ]:
# ---------------------------------------------------------------------
# Optional live ThreatCluster API integration
# ---------------------------------------------------------------------
import os
import time
import requests
from datetime import datetime, timezone
from functools import lru_cache
from langchain_core.documents import Document

THREATCLUSTER_API_BASE = "https://threatcluster.io/api/public/v1"
THREATCLUSTER_API_REFERENCE = "https://threatcluster.io/about/api-reference"
THREATCLUSTER_API_SEARCH_DOCS = "https://threatcluster.io/about/api-reference/threats"
_LIVE_API_CACHE = {}
_LIVE_API_CACHE_TTL_SECONDS = 600  # Cache repeated searches for 10 minutes to conserve credits.


def get_threatcluster_api_key():
    """Read API key from environment first, then Colab Secrets. Never print the key."""
    key = os.getenv("THREATCLUSTER_API_KEY")
    if key:
        return key.strip()
    try:
        from google.colab import userdata
        key = userdata.get("THREATCLUSTER_API_KEY")
        return key.strip() if key else None
    except Exception:
        return None


def _live_search_term(query):
    """Choose one narrow search token; never send a whole email or long user prompt."""
    text = str(query or "").strip()
    cve = re.search(r"CVE-\d{4}-\d{3,}", text, re.I)
    if cve:
        return cve.group(0)
    low = text.lower()
    specific_aliases = [("lockbit", "LockBit"), ("blackcat", "BlackCat"), ("alphv", "ALPHV"), ("miniduke", "MiniDuke")]
    for alias, canonical in specific_aliases:
        if alias in low:
            return canonical
    tokens = re.findall(r"[a-zA-Z0-9][a-zA-Z0-9._-]{2,}", text)
    stop = {
        "what", "which", "when", "where", "does", "this", "that", "with", "from",
        "about", "give", "show", "find", "tell", "how", "can", "could", "would",
        "should", "why", "is", "are", "describe", "discuss", "analyze", "analyse",
        "summarize", "summarise", "provide", "list", "identify", "name", "compare",
        "latest", "recent", "current", "known",
        "threat", "threats", "cyber", "attack", "attacks", "incident", "incidents",
        "security", "mitigation", "mitigations", "documented", "explain", "please",
        "dataset", "datasets", "report", "reports", "impact", "loss", "losses",
        "healthcare", "ransomware", "phishing", "vulnerability", "vulnerabilities",
        "case", "cases", "case-study", "studies", "study", "documented", "named",
        "today", "yesterday", "week", "this", "past", "last", "current", "landscape"
    }
    # Favor distinctive proper-name-like tokens before broad topic terms.
    distinctive = [t for t in tokens if t.lower() not in stop and (any(ch.isupper() for ch in t) or any(ch.isdigit() for ch in t))]
    if distinctive:
        return distinctive[0][:80]
    topic_map = [
        ("healthcare", ("healthcare", "hospital", "clinic", "medical")),
        ("ransomware", ("ransomware",)),
        ("phishing", ("phishing", "spearphishing")),
        ("wiper", ("wiper", "destructive malware")),
        ("apt", ("apt", "state-sponsored", "cyber espionage")),
    ]
    for term, signals in topic_map:
        if any(signal in low for signal in signals):
            return term
    candidates = [t for t in tokens if t.lower() not in stop]
    if candidates:
        return candidates[0][:80]
    if any(term in low for term in ("latest", "recent", "current", "today", "this week")):
        return "threat"
    return ""


def _live_api_document(item, bucket, fetched_at, request_cost, credits_left):
    """Turn one API hit into a traceable document. Preserve only fields actually returned."""
    item = item if isinstance(item, dict) else {"value": item}
    title = str(item.get("title") or item.get("name") or item.get("victim_name") or item.get("entity_value") or f"ThreatCluster {bucket} record")[:300]
    summary = item.get("summary") or item.get("description") or item.get("snippet") or ""
    entities = item.get("entities") or item.get("entity_names") or []
    if isinstance(entities, (dict, list)):
        entities_text = json.dumps(entities, ensure_ascii=False, default=str)[:1000]
    else:
        entities_text = str(entities)[:1000]
    date_value = (
        item.get("last_reported") or item.get("published_at") or item.get("first_reported")
        or item.get("created_at") or item.get("discovered_at") or item.get("date") or "Date not supplied"
    )
    record_id = str(item.get("short_id") or item.get("cluster_id") or item.get("id") or item.get("slug") or "Not supplied")[:120]
    score = item.get("threat_score")
    urgency = item.get("urgency") or item.get("urgency_level")
    source_links = item.get("sources") or item.get("articles") or []
    source_urls = []
    if isinstance(source_links, list):
        for source in source_links[:4]:
            if isinstance(source, dict):
                u = source.get("url") or source.get("source_url") or source.get("link")
                if u and str(u).startswith("http"):
                    source_urls.append(str(u))
            elif isinstance(source, str) and source.startswith("http"):
                source_urls.append(source)
    direct_url = item.get("url") or item.get("source_url") or item.get("primary_url")
    if direct_url and str(direct_url).startswith("http"):
        source_urls.insert(0, str(direct_url))
    fields = []
    if summary:
        fields.append(f"Summary: {str(summary)[:2600]}")
    if entities_text:
        fields.append(f"Entities returned by API: {entities_text}")
    if date_value:
        fields.append(f"Reported/record date returned by API: {date_value}")
    if score is not None:
        fields.append(f"Publisher threat score (not a probability): {score}")
    if urgency is not None:
        fields.append(f"Publisher urgency: {urgency}")
    # Include a compact field excerpt to retain bucket-specific information without dumping raw payloads.
    keep = {k: v for k, v in item.items() if k in {
        "entity_type", "entity_value", "category", "country", "sector", "group_name",
        "victim_name", "cve_id", "severity", "cvss_v3_score", "in_kev", "has_exploit",
        "threat_score", "urgency", "urgency_level", "first_reported", "last_reported"
    } and v not in (None, "", [], {})}
    if keep:
        fields.append("Additional returned fields: " + json.dumps(keep, ensure_ascii=False, default=str))
    if bucket == "darkweb":
        fields.append("Source-type caveat: this is a dark-web/leak-site record returned by the API; treat victim statements as claims unless independently corroborated.")
    if source_urls:
        fields.append("Original reporting/source URLs: " + " | ".join(dict.fromkeys(source_urls[:4])))
    fields.append(f"Live API fetched at (UTC): {fetched_at}")
    fields.append(f"API request cost: {request_cost if request_cost is not None else 'Not returned'}; credits remaining: {credits_left if credits_left is not None else 'Not returned'}")
    content = f"Live ThreatCluster API {bucket} record: {title}\n" + "\n".join(fields)
    return Document(
        page_content=content,
        metadata={
            "source_dataset": "threatcluster-live-api",
            "source_kind": "live_api",
            "record_type": f"live_{bucket}",
            "title": title,
            "record_id": record_id,
            "record_date": str(date_value),
            "source_url": source_urls[0] if source_urls else THREATCLUSTER_API_REFERENCE,
            "api_endpoint": "/search",
            "api_docs_url": THREATCLUSTER_API_SEARCH_DOCS,
            "fetched_at": fetched_at,
            "request_cost": str(request_cost) if request_cost is not None else "Not returned",
            "credits_remaining": str(credits_left) if credits_left is not None else "Not returned",
            "bucket": bucket,
        }
    )


def fetch_threatcluster_live(query, enabled=True, privacy_safe_term=None):
    """Search ThreatCluster live API with one keyword, 7-day window and a 10-minute cache."""
    if not enabled:
        return [], "Live API not requested for this query.", {}
    key = get_threatcluster_api_key()
    if not key:
        return [], "Live API not configured: add THREATCLUSTER_API_KEY to Colab Secrets or the Space secret settings.", {}
    search_term = str(privacy_safe_term or _live_search_term(query)).strip()
    if len(search_term) < 2:
        return [], "Live API skipped: no suitable two-character search term was extracted.", {}

    now = time.time()
    cache_key = search_term.lower()
    cached = _LIVE_API_CACHE.get(cache_key)
    if cached and (now - cached["cached_at"]) < _LIVE_API_CACHE_TTL_SECONDS:
        return cached["docs"], cached["status"] + " (cached; no new API credit used)", cached["meta"]

    headers = {"X-API-Key": key, "Accept": "application/json", "User-Agent": "CTI-RAG-Student-Project/1.0"}
    params = {"q": search_term, "limit": 5, "days": 7, "include_articles": "false"}
    try:
        response = requests.get(
            f"{THREATCLUSTER_API_BASE}/search", headers=headers, params=params, timeout=(5, 15)
        )
        request_cost = response.headers.get("X-Request-Cost")
        credits_left = response.headers.get("X-RateLimit-Remaining")
        response.raise_for_status()
        payload = response.json()
        fetched_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
        docs_out = []
        for bucket in ("clusters", "entities", "darkweb"):
            records = payload.get(bucket, []) if isinstance(payload, dict) else []
            if isinstance(records, list):
                for item in records[:5]:
                    docs_out.append(_live_api_document(item, bucket, fetched_at, request_cost, credits_left))
        docs_out = docs_out[:8]
        status = (
            f"Live ThreatCluster API searched `{search_term}` (last 7 days); "
            f"{len(docs_out)} records returned; request cost {request_cost or 'not supplied'} credit(s); "
            f"credits remaining today {credits_left or 'not supplied'}; fetched {fetched_at} UTC."
        )
        meta = {"query": search_term, "fetched_at": fetched_at, "request_cost": request_cost, "credits_remaining": credits_left, "days": 7}
        _LIVE_API_CACHE[cache_key] = {"cached_at": now, "docs": docs_out, "status": status, "meta": meta}
        return docs_out, status, meta
    except requests.HTTPError as exc:
        status = f"Live API request failed ({response.status_code}): {str(exc)[:300]}"
        if response.status_code == 403:
            status += " Check the API key's read scope and the free-tier seven-day lookback."
        elif response.status_code == 429:
            status += " The API rate or credit limit may have been reached."
        return [], status, {"http_status": response.status_code}
    except (requests.RequestException, ValueError) as exc:
        return [], f"Live API unavailable: {type(exc).__name__}: {str(exc)[:250]}", {}

print("ThreatCluster live API connector ready. Add THREATCLUSTER_API_KEY to enable it.")


# Evaluation & Testing

Run the evaluation cell below **before** launching the Gradio interface. It records retrieval latency, Gemma generation latency and total response time for historical, vulnerability and ransomware-target queries.

After reviewing each answer against its retrieved evidence, fill in the manual scores:
- **Correctness:** 0 = incorrect, 1 = partly correct, 2 = supported/correct.
- **Grounding:** 0 = unsupported, 1 = partly supported, 2 = all material claims supported by the records.
- **Relevance:** 0 = does not answer the question, 1 = partly relevant, 2 = directly relevant.

The CVE and ransomware sources are snapshots, not live databases. Do not invent or estimate evaluation scores, financial losses or breach confirmation.

Strict grounding check: verify that every case-specific statement, impact, financial-loss figure, and mitigation is present in the selected evidence. A missing field must be marked "Not found in the selected datasets or live API results." Do not award correctness for plausible facts supplied from model memory.

## Email-triage behaviour and safety boundary

The email workflow combines (1) transparent text indicators, (2) exact domain matching against the separately downloaded PhishDestroy feed, and (3) RAG retrieval for historical context. Domain matches are leads, not proof; non-matches do not mean safe. The feed is checked once when this notebook starts, so restart/re-run the feed-loading cell to refresh it.

The answer separates the likely attacker objective, identifiable target, potential impact and actual loss. Actual loss is left as unknown unless a source documents it. No links, attachments or scripts are opened or executed.

In [ ]:
import time
import pandas as pd

# These questions test historical, current-snapshot and structured-record retrieval.
latest_cve_id = ""
if not cve_df.empty and "cve_id" in cve_df.columns:
    latest_cve_rows = cve_df.sort_values("published_date", ascending=False) if "published_date" in cve_df.columns else cve_df
    latest_cve_id = str(latest_cve_rows.iloc[0]["cve_id"])

latest_cve_question = (
    f"For {latest_cve_id}, what does the loaded snapshot say about severity and exploitation?"
    if latest_cve_id else
    "What vulnerability fields are available in the loaded CVE snapshot?"
)

evaluation_questions = [
    "What do historical reports state about MiniDuke PDF zero-day attacks and its micro backdoor?",
    latest_cve_question,
    "What ransomware-group victim claims are recorded for healthcare organisations, and what does the dataset actually establish?"
]

evaluation_rows = []
for question in evaluation_questions:
    retrieval_start = time.perf_counter()
    retrieved = hybrid_retrieve(question)
    structured = retrieve_structured_context(question)
    case_docs = retrieve_incident_case_studies(question, max_cases=2)
    case_context = format_docs(case_docs, max_chars_per_doc=1400) if case_docs else "No matching named incident case was found in the loaded incident summaries."
    retrieval_seconds = time.perf_counter() - retrieval_start

    rag_start = time.perf_counter()
    rag_answer = qa_chain.invoke({
        "context": format_docs(retrieved),
        "structured_context": structured,
        "case_study_context": case_context,
        "input": question,
        "response_instruction": "QUESTION TYPE: Direct CTI Q&A. Answer the exact question in a concise paragraph or bullets. Cite every factual claim and mark missing facts as 'Not found in the selected datasets or live API results.'"
    })
    generation_seconds = time.perf_counter() - rag_start

    evaluation_rows.append({
        "question": question,
        "retrieved_chunks": len(retrieved),
        "incident_cases_selected": len(case_docs),
        "retrieval_seconds": round(retrieval_seconds, 3),
        "generation_seconds": round(generation_seconds, 3),
        "total_seconds": round(retrieval_seconds + generation_seconds, 3),
        "answer": rag_answer,
        "human_correctness_0_to_2": None,
        "grounding_0_to_2": None,
        "review_notes": "Manually verify each claim against the cited source records."
    })

evaluation_results = pd.DataFrame(evaluation_rows)
display(evaluation_results)
evaluation_results.to_csv("cti_multisource_evaluation_results.csv", index=False)
print("Saved cti_multisource_evaluation_results.csv")

## Response-speed and progress improvements

- The first query is warmed up when Gemma loads, so one-time CUDA initialisation is less likely to affect the first analyst request.
- Gemma uses KV caching, greedy decoding, one beam and a 210-token response cap. Increase `MAX_NEW_TOKENS` to 140 if answers are cut off; this may increase latency.
- Email triage retrieves with a short intent query and skips CVE/ransomware table scans that rarely help classify a pasted email.
- Structured table matching uses one vectorised regex pass instead of a separate full-table scan for every term.
- Gradio shows a progress indicator and status descriptions while the request runs. Requests are serialised to avoid GPU contention.
- Measure retrieval and generation separately. The actual latency depends on the selected Colab GPU and current runtime load.


## Named incident case studies

For broad sector/attack questions (for example healthcare ransomware), the app now selects up to two distinct incident summaries from the incident-cluster dataset. It separates case-specific facts, documented response, actual losses and recommended mitigations. Ransomware leak-site victim rows are not treated as confirmed incidents. If the incident snapshot has no matching named cases, the app must say so rather than inventing examples.


## Approved-source factuality / abstention
- CTI research answers may only use supplied records from the selected datasets and, when enabled, the separately labelled live ThreatCluster API.
- The notebook includes a pre-generation relevance gate. When there is no sufficiently matching evidence, it returns a fixed abstention response without calling Gemma.
- For broad sector + attack questions, named cases must come from the selected incident-summary dataset. If no matching named case is found, the assistant says so and does not import famous cases from general knowledge.
- Within an otherwise relevant case, unavailable target, method, impact, loss, response, and mitigation fields are each reported as "Not found in the selected datasets or live API results." 
- The prompt forbids generic mitigation advice from model memory. Mitigation is reported only when a selected dataset explicitly contains it.
- For email triage, the submitted email is treated as direct input evidence for visible warning signs. Campaign attribution, threat-actor names, confirmed incidents, losses, and mitigation must still be supported by selected dataset records or the downloaded domain feed.


In [ ]:
# ---------------------------------------------------------------------
# Gradio user interface
# ---------------------------------------------------------------------
import gradio as gr
import time
import re

def _source_record_markdown(doc, label):
    meta = doc.metadata or {}
    if meta.get("source_kind") == "live_api":
        lines = [
            f"### [{label}] {meta.get('title', 'Live ThreatCluster record')}",
            "**Source channel:** ThreatCluster live REST API (not a static Hugging Face dataset)",
            f"**API endpoint:** `{meta.get('api_endpoint', '/search')}`",
            f"**Record ID:** {meta.get('record_id', 'Not supplied')}",
            f"**Record date returned by API:** {meta.get('record_date', 'Not supplied')}",
            f"**Fetched at (UTC):** {meta.get('fetched_at', 'Not supplied')}",
            f"**Request cost:** {meta.get('request_cost', 'Not returned')} credit(s)",
            f"**Credits remaining:** {meta.get('credits_remaining', 'Not returned')}",
            f"**API reference:** [{meta.get('api_docs_url', THREATCLUSTER_API_REFERENCE)}]({meta.get('api_docs_url', THREATCLUSTER_API_REFERENCE)})",
        ]
        if meta.get('source_url') and meta.get('source_url') != THREATCLUSTER_API_REFERENCE:
            lines.append(f"**Original report/source:** [{meta['source_url']}]({meta['source_url']})")
        lines.append("**Retrieved API evidence:**\n\n" + (doc.page_content or ""))
        return "\n\n".join(lines)
    dataset_name = meta.get("source_dataset", "Unknown dataset")
    info = _dataset_info(dataset_name)
    title = meta.get("title", "Untitled record")
    date = meta.get("record_date", "Date not supplied")
    record_id = meta.get("record_id", "Not supplied")
    original_url = meta.get("source_url", "")
    if original_url == info.get("url"):
        original_url = ""
    lines = [
        f"### [{label}] {title}",
        f"**Dataset:** `{dataset_name}`",
        f"**Dataset citation:** {info['owner']}. ({info['year']}). *{info['title']}* [Data set]. Hugging Face.",
        f"**Dataset card:** [{info['url']}]({info['url']})" if info.get("url") else "**Dataset card:** Not supplied",
        f"**Record date:** {date}",
        f"**Record ID:** {record_id}",
        f"**Dataset note:** {info.get('note', '')}",
        f"**License shown on dataset card:** {info.get('license', 'Check dataset card')}"
    ]
    if original_url:
        lines.append(f"**Original report/article:** [{original_url}]({original_url})")
    lines.append("**Retrieved excerpt:**\n\n" + (doc.page_content or ""))
    return "\n\n".join(lines)


def build_sources_panel(retrieved_docs, structured_context, case_docs=None, include_phish_feed=False, live_docs=None, live_status=""):
    sections = []
    if live_status:
        sections.append("### Live API status\n\n" + live_status)
    for index, doc in enumerate(live_docs or [], start=1):
        sections.append(_source_record_markdown(doc, f"L{index}"))
    for index, doc in enumerate(case_docs or [], start=1):
        sections.append(_source_record_markdown(doc, f"C{index}"))
    for index, doc in enumerate(retrieved_docs or [], start=1):
        sections.append(_source_record_markdown(doc, f"S{index}"))

    if structured_context and structured_context.strip() not in {
        _NO_STRUCTURED, "Not applicable to this email-only check."
    }:
        sections.append("### Structured selected-dataset records\n\n" + structured_context)

    if include_phish_feed:
        info = _dataset_info("phishdestroy/destroylist")
        sections.append(
            "### [IOC1] PhishDestroy domain-feed source\n"
            "**Dataset citation:** phishdestroy. (n.d.). *destroylist* [Data set]. Hugging Face.\n\n"
            f"**Dataset card:** [{info['url']}]({info['url']})\n\n"
            f"**Feed timestamp reported by publisher:** {IOC_FEED_LAST_UPDATED}\n\n"
            f"**Feed status:** {IOC_FEED_STATUS}"
        )
    if not sections:
        return "No supporting records were retrieved from the selected datasets."
    return "\n\n---\n\n".join(sections)


def build_answer_citations(retrieved_docs, case_docs=None, structured_context="", include_phish_feed=False, live_docs=None):
    lines = []
    for index, doc in enumerate(live_docs or [], start=1):
        meta = doc.metadata or {}
        source_url = meta.get("source_url", THREATCLUSTER_API_REFERENCE)
        item = f"- **[L{index}] {meta.get('title', 'Live ThreatCluster record')}** — ThreatCluster live API result (fetched {meta.get('fetched_at', 'time not supplied')} UTC); [API reference]({meta.get('api_docs_url', THREATCLUSTER_API_REFERENCE)})"
        if source_url and source_url != THREATCLUSTER_API_REFERENCE:
            item += f"; [original report/source]({source_url})"
        lines.append(item)
    for index, doc in enumerate(case_docs or [], start=1):
        label = f"C{index}"
        meta = doc.metadata or {}
        dataset_name = meta.get("source_dataset", "Unknown dataset")
        info = _dataset_info(dataset_name)
        original = meta.get("source_url", "")
        if original == info.get("url"):
            original = ""
        citation = f"{info['owner']}. ({info['year']}). *{info['title']}* [Data set]. Hugging Face."
        item = f"- **[{label}] {meta.get('title', 'Untitled record')}** — {citation} [Dataset card]({info['url']})"
        if original:
            item += f"; [record/original report]({original})"
        lines.append(item)
    for index, doc in enumerate(retrieved_docs or [], start=1):
        label = f"S{index}"
        meta = doc.metadata or {}
        dataset_name = meta.get("source_dataset", "Unknown dataset")
        info = _dataset_info(dataset_name)
        original = meta.get("source_url", "")
        if original == info.get("url"):
            original = ""
        citation = f"{info['owner']}. ({info['year']}). *{info['title']}* [Data set]. Hugging Face."
        item = f"- **[{label}] {meta.get('title', 'Untitled record')}** — {citation} [Dataset card]({info['url']})"
        if original:
            item += f"; [record/original report]({original})"
        lines.append(item)
    for label in dict.fromkeys(re.findall(r"\[(V\d+)\]", structured_context or "")):
        info = _dataset_info("threatcluster/cve-exploitation-signals")
        citation = f"{info['owner']}. ({info['year']}). *{info['title']}* [Data set]. Hugging Face."
        lines.append(f"- **[{label}] CVE structured record** — {citation} [Dataset card]({info['url']})")
    for label in dict.fromkeys(re.findall(r"\[(R\d+)\]", structured_context or "")):
        info = _dataset_info("threatcluster/ransomware-leak-site-victims")
        citation = f"{info['owner']}. ({info['year']}). *{info['title']}* [Data set]. Hugging Face."
        lines.append(f"- **[{label}] Ransomware listing record** — {citation} [Dataset card]({info['url']})")
    if include_phish_feed:
        info = _dataset_info("phishdestroy/destroylist")
        citation = f"{info['owner']}. ({info['year']}). *{info['title']}* [Data set]. Hugging Face."
        lines.append(f"- **[IOC1] Domain-feed check** — {citation} [Dataset card]({info['url']})")
    if not lines:
        return "\n\n### Dataset citations\nNot found in the selected datasets or live API results. No source record was retrieved."
    return "\n\n### Dataset citations and retrieved records\n" + "\n".join(lines)


_NOT_FOUND = "Not found in the selected datasets or live API results."
_NO_STRUCTURED = "No matching structured CVE or ransomware-target records were selected for this query."


def query_response_style(query):
    """Distinguish explicit requests for named cases from ordinary CTI explanations/comparisons."""
    low = str(query or "").lower()
    explicit_case_request = bool(re.search(
        r"\b(case study|case studies|documented cases?|real[- ]world case(?:s| study| studies)?|"
        r"similar real[- ]world case examples?|real[- ]world example(?:s)?|similar case(?:s)?|"
        r"similar incident(?:s)?|give (?:me )?(?:one|two|three|some) (?:cases?|incidents?|examples?))\b",
        low, re.I
    ))
    has_sector = any(
        re.search(pattern, low, re.I)
        for patterns in _CASE_SECTOR_PATTERNS.values()
        for pattern in patterns
    )
    has_attack = any(
        re.search(pattern, low, re.I)
        for patterns in _CASE_ATTACK_PATTERNS.values()
        for pattern in patterns
    )
    if explicit_case_request or (has_sector and has_attack):
        return "case_studies"

    asks_definition = bool(re.search(
        r"\b(what is|what are|define|definition of|explain|meaning of)\b", low, re.I
    ))
    asks_mitigation = bool(re.search(
        r"\b(mitigat\w*|prevent\w*|protect\w*|defen[cs]\w*|countermeasure\w*)\b",
        low, re.I
    ))
    if asks_definition and asks_mitigation:
        return "definition_mitigation"
    if asks_definition:
        return "definition"
    if asks_mitigation:
        return "mitigation"
    return "direct_answer"


def response_instruction_for(query):
    style = query_response_style(query)
    if style == "definition_mitigation":
        return (
            "QUESTION TYPE: Definition plus mitigation. Do NOT show documented cases, Case 1, or Case 2. "
            "Use this structure only: ## Direct answer (one or two sentences defining the concept from the records); "
            "## Mitigation documented in the selected datasets (bullet only actions explicitly present in records; otherwise write 'Not found in the selected datasets or live API results.'). "
            "Do not provide generic advice from model memory. Cite every fact with the exact [S#], [C#], [V#], [R#], or [L#] evidence label."
        )
    if style == "definition":
        return "QUESTION TYPE: Definition. Do NOT show cases. Give a direct, concise definition supported by records, cite each fact, and state missing details as 'Not found in the selected datasets or live API results.'"
    if style == "mitigation":
        return "QUESTION TYPE: Mitigation. Do NOT show cases unless explicitly requested. List only mitigations explicitly recorded in supplied data; otherwise say 'Not found in the selected datasets or live API results.' Cite each action."
    if style == "case_studies":
        return (
            "QUESTION TYPE: Case study / real-world example. Prefer named incident-cluster records [C#] or live records [L#]. "
            "If those are absent, you may use a historical report or curated CTI record [S#] only when it directly names and describes the requested case. "
            "Do not invent a similar case from model memory. Explain the sequence only to the level documented in the supplied evidence. "
            "For missing target, impact, response or mitigation details say 'Not found in the selected datasets or live API results.' "
            "Cite every factual statement with the exact [C#], [S#], or [L#] label."
        )
    return "QUESTION TYPE: Direct CTI Q&A. Answer the exact question in a concise paragraph or bullets. Do NOT show cases unless requested. Cite every factual claim and mark missing facts as 'Not found in the selected datasets or live API results.'"


def has_relevant_selected_evidence(query, retrieved_docs, structured_context, case_docs=None, live_docs=None):
    """
    Evidence gate for CTI Q&A.

    It does not require a named incident case for a definition or comparison.
    It does require retrieved evidence that overlaps with the requested subject.
    Named CVEs and named malware/campaigns are checked explicitly.
    """
    q = str(query or "").strip()
    if not q:
        return False, []

    docs_all = list(case_docs or []) + list(retrieved_docs or []) + list(live_docs or [])
    parts = [(doc.page_content or "").lower() for doc in docs_all]
    if structured_context and structured_context.strip() not in {
        _NO_STRUCTURED, "Not applicable to this email-only check."
    }:
        parts.append(str(structured_context).lower())
    evidence = "\n".join(parts)
    if not evidence.strip():
        return False, []

    # Exact CVE rule: every requested CVE must occur in selected evidence.
    requested_cves = [c.upper() for c in re.findall(r"\bCVE-\d{4}-\d{3,}\b", q, re.I)]
    if requested_cves:
        missing = [c for c in requested_cves if c.lower() not in evidence]
        if missing:
            return False, []
        return True, requested_cves

    # Named malware/campaign names must be present in evidence. This prevents
    # pretrained model memory from being used to invent a MiniDuke case.
    named_terms = []
    for phrase in ("miniduke", "mini duke", "apt29", "cozy bear", "solarwinds",
                   "wannacry", "notpetya", "not petya", "stuxnet", "conficker",
                   "emotet", "trickbot", "lockbit", "conti", "ryuk"):
        if re.search(rf"(?<![a-z0-9]){re.escape(phrase)}(?![a-z0-9])", q.lower()):
            named_terms.append(phrase)
    if named_terms and not any(term in evidence for term in named_terms):
        return False, []

    # Concept aliases handle common CTI terminology variants.
    concept_groups = {
        "command_and_control": [
            "command-and-control", "command and control", "command control",
            "c2", "c&c", "c2 channel", "c2 communication", "command channel"
        ],
        "obfuscation": [
            "obfuscation", "obfuscated", "obfuscate", "packing", "packed",
            "encoding", "encoded", "masquerading", "steganography"
        ],
        "encryption": [
            "encryption", "encrypted", "cryptographic", "cipher", "tls", "ssl",
            "https", "aes", "rc4", "rsa", "chacha20"
        ],
        "network_protocol": [
            "protocol", "protocols", "dns", "http", "https", "tls", "ssl",
            "tcp", "udp", "websocket", "ssh", "icmp"
        ],
        "zero_day": [
            "zero-day", "zero day", "0-day", "0 day", "unpatched", "CVE-2013-0640"
        ],
        "pdf_exploit": [
            "pdf", "adobe reader", "acrobat", "malicious document", "document exploit"
        ],
        "ransomware": [
            "ransomware", "file encryption", "double extortion", "data extortion"
        ],
        "phishing": [
            "phishing", "spear phishing", "credential theft", "social engineering"
        ],
        "firewall": [
            "firewall", "network filtering", "packet filtering", "stateful inspection"
        ],
    }

    matched_groups = []
    q_low = q.lower()
    for group, aliases in concept_groups.items():
        q_has_group = any(
            re.search(rf"(?<![a-z0-9]){re.escape(alias.lower())}(?![a-z0-9])", q_low)
            for alias in aliases
        )
        if not q_has_group:
            continue
        evidence_has_group = any(
            re.search(rf"(?<![a-z0-9]){re.escape(alias.lower())}(?![a-z0-9])", evidence)
            for alias in aliases
        )
        if evidence_has_group:
            matched_groups.append(group)

    # General lexical anchors, excluding instruction words. Keep the rule
    # proportional so comparisons with several related concepts are not rejected.
    generic = set(_STOP_WORDS) | {
        "case", "cases", "study", "studies", "example", "examples", "similar",
        "actual", "reported", "reporting", "known", "using", "dataset", "selected",
        "organisation", "organisations", "organization", "organizations",
        "please", "could", "would", "should", "state", "states", "regarding",
        "information", "compare", "comparison", "techniques", "technique",
        "protocol", "protocols", "methods", "method", "explain", "explains",
        "explanation", "works", "work", "step", "steps", "detail", "details",
        "list", "give", "describe", "what", "how", "why", "which", "the"
    }
    tokens = [
        t for t in re.findall(r"[a-z0-9][a-z0-9._-]{1,}", q_low)
        if t not in generic and not t.isdigit()
    ]
    matched_tokens = [
        t for t in dict.fromkeys(tokens)
        if re.search(rf"(?<![a-z0-9]){re.escape(t)}(?![a-z0-9])", evidence)
    ]

    # A matched concept family is enough for broad technical questions such as
    # "Compare C2 obfuscation and encryption protocols". For other questions,
    # require at least one meaningful lexical anchor; two for long compound queries.
    if matched_groups:
        return True, matched_groups + matched_tokens[:8]

    if not tokens:
        return False, []
    required = 1 if len(tokens) <= 3 else 2
    if len(matched_tokens) >= required:
        return True, matched_tokens
    return False, matched_tokens


def dataset_only_no_answer(reason, retrieval_seconds=0.0, total_seconds=0.0):
    return (
        "## No supported answer found\n\n"
        f"**{_NOT_FOUND}**\n\n{reason}\n\n"
        "The application did not use outside knowledge to fill the gap. "
        "Try a more specific question containing an incident name, organisation, threat actor, malware family, CVE identifier, or attack technique.\n\n"
        f"*Processing: retrieval {retrieval_seconds:.2f}s · total {total_seconds:.2f}s.*"
    )


def process_soc_query(user_query, analysis_mode, use_live_api=True, live_search_term=None, progress=gr.Progress(track_tqdm=False)):
    if not isinstance(user_query, str) or not user_query.strip():
        return "Please enter a CTI question or paste the email text.", "No input was submitted."
    total_start = time.perf_counter()
    try:
        progress(0.05, desc="Preparing your request...")
        findings, rule_assessment, case_docs, ioc_lines = [], "", [], []
        response_style = query_response_style(user_query) if analysis_mode != "Email triage" else "email_triage"
        if analysis_mode == "Email triage":
            findings = detect_email_indicators(user_query)
            rule_assessment = classify_email_preliminary(user_query, findings)
            retrieval_query = "phishing email social engineering credential theft malicious link attachment " + " ".join(findings)
        else:
            retrieval_query = user_query[:1500]

        progress(0.18, desc="Searching selected datasets and live intelligence...")
        retrieval_start = time.perf_counter()
        if analysis_mode == "Email triage":
            retrieved_docs = hybrid_retrieve(retrieval_query, k=2, candidate_k=5)
            structured_context = "Not applicable to this email-only check."
        else:
            retrieved_docs = hybrid_retrieve(retrieval_query, k=TOP_K, candidate_k=CANDIDATE_K)
            structured_context = retrieve_structured_context(user_query)
            if response_style == "case_studies":
                case_docs = retrieve_incident_case_studies(user_query, max_cases=2)
        if analysis_mode == "CTI research":
            live_docs, live_status, live_meta = fetch_threatcluster_live(
                user_query, enabled=use_live_api, privacy_safe_term=live_search_term
            )
        else:
            # Do not send the email body, sender, or full URL to the live API.
            live_docs, live_status, live_meta = [], "Live API enrichment is disabled until a domain-only lookup is selected below.", {}
        live_case_docs = [d for d in live_docs if (d.metadata or {}).get("bucket") == "clusters"]
        retrieval_seconds = time.perf_counter() - retrieval_start

        context = format_docs(retrieved_docs) if retrieved_docs else "No directly relevant narrative CTI excerpts were retrieved."
        if live_docs:
            context += "\n\nLIVE THREATCLUSTER API RECORDS (distinct live source channel):\n" + format_docs(live_docs, max_chars_per_doc=900, label_prefix="L")
        case_study_sections = []
        if case_docs:
            case_study_sections.append(format_docs(case_docs, max_chars_per_doc=1400, label_prefix="C"))
        if live_case_docs:
            case_study_sections.append(format_docs(live_case_docs, max_chars_per_doc=1400, label_prefix="L"))
        case_study_context = (
            "\n\n".join(case_study_sections)
            if case_study_sections else "Case studies were not requested for this question. Do not output Case 1, Case 2, or named incidents."
        )
        # A named case row is preferred, but it is not the only valid evidence source.
        # Historical reports and curated CTI records may support a case explanation.
        # Refuse only when neither named cases nor relevant narrative/live evidence exists.
        if (analysis_mode != "Email triage" and response_style == "case_studies"
                and not case_docs and not live_case_docs and not retrieved_docs):
            total_seconds = time.perf_counter() - total_start
            answer = dataset_only_no_answer(
                "No matching named case study or supporting historical report was retrieved.",
                retrieval_seconds, total_seconds
            )
            progress(1.0, desc="No supporting case evidence found.")
            return answer, build_sources_panel(
                retrieved_docs, structured_context, case_docs=case_docs,
                live_docs=live_docs, live_status=live_status
            )

        if analysis_mode != "Email triage":
            evidence_ok, _ = has_relevant_selected_evidence(user_query, retrieved_docs, structured_context, case_docs, live_docs)
            if not evidence_ok:
                total_seconds = time.perf_counter() - total_start
                answer = dataset_only_no_answer(
                    "The retrieved records did not contain enough directly matching evidence.",
                    retrieval_seconds, total_seconds
                )
                progress(1.0, desc="Insufficient matching evidence.")
                return answer, build_sources_panel(retrieved_docs, structured_context, case_docs=case_docs, live_docs=live_docs, live_status=live_status)

        progress(0.42, desc="Checking indicators and domain reputation...")
        if analysis_mode == "Email triage":
            lookup_start = time.perf_counter()
            ioc_results = check_email_domains(user_query)
            # Privacy boundary: live API enrichment receives only the first extracted domain,
            # never the pasted email body, headers, sender name or complete URL path.
            privacy_domain = ioc_results[0]["domain"] if ioc_results else None
            if not use_live_api:
                live_docs, live_status, live_meta = [], "Live API not requested for this email.", {}
            elif privacy_domain:
                live_docs, live_status, live_meta = fetch_threatcluster_live(
                    user_query, enabled=True, privacy_safe_term=privacy_domain
                )
                if live_docs:
                    context += "\n\nLIVE THREATCLUSTER API RECORDS (domain-only lookup):\n" + format_docs(live_docs, max_chars_per_doc=900, label_prefix="L")
            else:
                live_docs, live_status, live_meta = [], "Live API skipped for email triage: no domain was extracted; email text was not sent.", {}
            retrieval_seconds += time.perf_counter() - lookup_start
            if not ioc_results:
                ioc_lines.append("No URL was supplied, so no domain-feed lookup was performed.")
            else:
                for result in ioc_results:
                    if result["matches"]:
                        ioc_lines.append(f"MATCH: {result['domain']} appears in the downloaded blocklist: {', '.join(result['matches'])}")
                    else:
                        ioc_lines.append(f"NO MATCH: {result['domain']} was not found in the downloaded feed; this is not proof of safety.")
            if not findings and not ioc_results and not retrieved_docs:
                total_seconds = time.perf_counter() - total_start
                answer = dataset_only_no_answer(
                    "No warning signs, domain-feed matches, or relevant CTI records were found. This is not proof that the email is safe.",
                    retrieval_seconds, total_seconds
                )
                progress(1.0, desc="No supporting indicators found.")
                return answer, build_sources_panel(retrieved_docs, structured_context, case_docs=case_docs, live_docs=live_docs, live_status=live_status)

        progress(0.58, desc="Generating a source-bound answer with Gemma...")
        generation_start = time.perf_counter()
        if analysis_mode == "Email triage":
            answer = email_triage_chain.invoke({
                "indicators": "\n".join(f"- {x}" for x in findings),
                "ioc_results": "\n".join(ioc_lines),
                "feed_timestamp": IOC_FEED_LAST_UPDATED,
                "feed_status": IOC_FEED_STATUS,
                "email_text": user_query[:6000],
                "context": context + "\n\nLive API status: " + live_status,
                "rule_assessment": rule_assessment
            })
        else:
            answer = qa_chain.invoke({
                "context": context,
                "structured_context": structured_context,
                "case_study_context": case_study_context,
                "input": user_query[:1500],
                "response_instruction": response_instruction_for(user_query)
            })
        generation_seconds = time.perf_counter() - generation_start
        total_seconds = time.perf_counter() - total_start
        answer += build_answer_citations(
            retrieved_docs, case_docs, structured_context,
            include_phish_feed=(analysis_mode == "Email triage" and bool(ioc_results)),
            live_docs=live_docs
        )
        answer += (
            f"\n\n---\n*Processing: retrieval {retrieval_seconds:.2f}s · "
            f"Gemma generation {generation_seconds:.2f}s · total {total_seconds:.2f}s.*"
        )
        print(f"CTI timing | retrieval={retrieval_seconds:.2f}s | generation={generation_seconds:.2f}s | total={total_seconds:.2f}s")
        progress(1.0, desc="Analysis complete.")
        return answer, build_sources_panel(
            retrieved_docs, structured_context, case_docs=case_docs,
            include_phish_feed=(analysis_mode == "Email triage" and bool(ioc_results)),
            live_docs=live_docs, live_status=live_status
        )
    except Exception as exc:
        print(f"CTI application error: {type(exc).__name__}: {exc}")
        return f"**System error:** {type(exc).__name__}: {exc}", "Check the notebook output for the technical traceback."


def _clear_outputs():
    return "", "", ""

custom_css = """
:root { color-scheme: dark; }
body, .gradio-container { background: #0b1220 !important; color: #e5e7eb !important; }
.gradio-container { max-width: 1180px !important; }
#hero { background: linear-gradient(120deg,#101b2e,#132b35); border: 1px solid #234454; border-radius: 16px; padding: 22px; }
#hero h1 { color: #65f0c2; margin-bottom: 6px; }
#hero p { color: #cbd5e1; }
#analyze-btn { min-height: 48px; font-weight: 700; }
#result-panel { border: 1px solid #26364a; border-radius: 12px; }
.muted { color: #94a3b8; font-size: 0.9rem; }
"""

app_theme = gr.themes.Soft(primary_hue="emerald", neutral_hue="slate")

with gr.Blocks(title="CTI RAG Assistant") as demo:
    gr.Markdown(
        """
        <div id="hero">
        <h1>🛡️ Cyber Threat Intelligence Assistant</h1>
        <p>Evidence-grounded CTI research and preliminary phishing-email triage.</p>
        <p class="muted">Answers use selected dataset records and separately labelled live API records where enabled; submitted email text is only used for observable indicators. Unsupported facts are reported as “Not found in the selected datasets or live API results.”</p>
        </div>
        """,
        elem_id="hero"
    )
    with gr.Row():
        gr.Markdown(
            """
            **Knowledge coverage**
            - Historical reports and periodic snapshots
            - Structured CVE and ransomware records
            - Curated CTI Q&A and PhishDestroy domain feed
            - Optional live ThreatCluster incident/entity search
            """
        )
        gr.Markdown(
            """
            **Data freshness**
            - Incident/CVE/ransomware collections: periodic snapshots
            - Historical reports and curated Q&A: archive / curated records
            - PhishDestroy: publisher describes an hourly-updated feed
            - ThreatCluster live API: queried at request time when enabled and keyed
            """
        )
    with gr.Tabs():
        with gr.Tab("🔎 CTI Research"):
            gr.Markdown("Ask about threat actors, malware, vulnerabilities, incidents, targets, impacts and documented mitigations.")
            cti_live_toggle = gr.Checkbox(value=True, label="Include live ThreatCluster API evidence (keyword search only; requires API key)")
            cti_input = gr.Textbox(
                label="CTI research question",
                placeholder="Example: Give two documented healthcare ransomware cases and explain their impact and documented response.",
                lines=5
            )
            with gr.Row():
                cti_submit = gr.Button("Analyse CTI question", variant="primary", elem_id="analyze-btn")
                cti_clear = gr.Button("Clear")
            cti_output = gr.Markdown(elem_id="result-panel")
            with gr.Accordion("Sources, dataset citations and retrieved evidence", open=False):
                cti_sources = gr.Markdown()
            gr.Examples(
                examples=[
                    ["What do historical reports state about MiniDuke PDF zero-day attacks and its micro backdoor?"],
                    ["What is a wiper attack and what mitigations are documented in the selected datasets?"],
                    ["Give two documented healthcare ransomware incident cases. Include actual losses only when reported."]
                ],
                inputs=cti_input
            )
        with gr.Tab("✉️ Email Triage"):
            gr.Markdown("Paste the email subject, sender, body and visible URLs. Remove confidential or personal information before submitting.")
            email_live_toggle = gr.Checkbox(value=False, label="Optional: send extracted domain only to ThreatCluster live API (email text is never sent)")
            email_input = gr.Textbox(
                label="Email content",
                placeholder="Subject: ...\nFrom: ...\n\nPaste the email body and URLs here...",
                lines=9
            )
            with gr.Row():
                email_submit = gr.Button("Analyse email", variant="primary", elem_id="analyze-btn")
                email_clear = gr.Button("Clear")
            email_output = gr.Markdown(elem_id="result-panel")
            with gr.Accordion("Email indicators, feed status and supporting CTI evidence", open=False):
                email_sources = gr.Markdown()
            gr.Examples(
                examples=[
                    ["Subject: Urgent account suspension\nFrom: security@example-mail.com\n\nYour account will be closed today. Verify your password immediately using https://example.invalid/verify."]
                ],
                inputs=email_input
            )

    def run_cti_query(q, live_toggle, progress=gr.Progress(track_tqdm=False)):
        return process_soc_query(q, "CTI research", use_live_api=live_toggle, progress=progress)

    def run_email_triage(q, live_toggle, progress=gr.Progress(track_tqdm=False)):
        return process_soc_query(q, "Email triage", use_live_api=live_toggle, progress=progress)

    cti_submit.click(
        fn=run_cti_query,
        inputs=[cti_input, cti_live_toggle], outputs=[cti_output, cti_sources],
        show_progress="full", concurrency_limit=1, concurrency_id="gemma-gpu"
    )
    email_submit.click(
        fn=run_email_triage,
        inputs=[email_input, email_live_toggle], outputs=[email_output, email_sources],
        show_progress="full", concurrency_limit=1, concurrency_id="gemma-gpu"
    )
    cti_clear.click(_clear_outputs, outputs=[cti_input, cti_output, cti_sources])
    email_clear.click(_clear_outputs, outputs=[email_input, email_output, email_sources])

demo.queue(default_concurrency_limit=1, max_size=16)
# Hugging Face Spaces supplies SPACE_ID and expects the app to bind to 0.0.0.0:7860.
if os.getenv("SPACE_ID"):
    demo.launch(theme=app_theme, css=custom_css, server_name="0.0.0.0", server_port=7860, share=False, debug=False)
else:
    demo.launch(theme=app_theme, css=custom_css, share=False, debug=False)


# Extended Evaluation: RAG vs. No-Retrieval Baseline (Ablation)

The same Gemma 2 2B model answers each question twice: once through the full RAG pipeline (`process_soc_query`) and once with no retrieved evidence. Comparing latency, citation use and abstention isolates what retrieval contributes. The question set covers historical, vulnerability, ransomware, definition/mitigation, case-study and out-of-scope questions; the out-of-scope question tests whether the system abstains instead of hallucinating.

Automatic columns (latency, citation count, abstention) are measured. Correctness, grounding and relevance must be scored manually (0–2) after checking each claim against the cited records. Do not invent scores.


In [ ]:
# ---------------------------------------------------------------------
# Extended evaluation: full RAG pipeline vs. no-retrieval Gemma baseline
# ---------------------------------------------------------------------
# Run after the Gradio cell (it reuses process_soc_query). The baseline sends the
# same question to the same Gemma 2 2B model WITHOUT retrieved evidence, so the
# difference isolates the contribution of retrieval (an ablation).
import re
import time
import pandas as pd

EVAL_QUESTIONS = [
    ("historical", "What do historical reports state about MiniDuke PDF zero-day attacks and its micro backdoor?"),
    ("vulnerability", latest_cve_question if "latest_cve_question" in globals() else "What vulnerability fields are available in the loaded CVE snapshot?"),
    ("ransomware", "What ransomware-group victim claims are recorded for healthcare organisations, and what does the dataset actually establish?"),
    ("definition_mitigation", "What is a wiper attack and what mitigations are documented in the selected datasets?"),
    ("case_studies", "Give two documented healthcare ransomware incident cases. Include actual losses only when reported."),
    ("out_of_scope", "What was the exact financial loss of the 2031 Mars colony ransomware attack?"),
]

BASELINE_TEMPLATE = (
    "<start_of_turn>user\nYou are a Cyber Threat Intelligence analyst. Answer concisely.\n\n"
    "{question}<end_of_turn>\n<start_of_turn>model\n"
)
CITATION_PATTERN = re.compile(r"\[(?:S|C|V|R|L)\d+\]")
NOT_FOUND = "not found in the selected datasets or live api results"
RETRIEVAL_PATTERN = re.compile(r"retrieval\s+([\d.]+)s")
GENERATION_PATTERN = re.compile(r"Gemma generation\s+([\d.]+)s")


def _no_progress(*args, **kwargs):
    return None


def _answer_body(answer):
    # Strip the appended citation list and timing footer so only Gemma's text is scored.
    return re.split(r"\n\n### Dataset citations|\n\n---\n\*Processing", answer or "", maxsplit=1)[0]


ablation_rows = []
for category, question in EVAL_QUESTIONS:
    rag_start = time.perf_counter()
    rag_answer, _sources = process_soc_query(question, "CTI research", use_live_api=False, progress=_no_progress)
    rag_seconds = time.perf_counter() - rag_start
    retrieval_match = RETRIEVAL_PATTERN.search(rag_answer or "")
    generation_match = GENERATION_PATTERN.search(rag_answer or "")

    base_start = time.perf_counter()
    baseline_answer = llm.invoke(BASELINE_TEMPLATE.format(question=question))
    baseline_seconds = time.perf_counter() - base_start

    rag_body = _answer_body(rag_answer)
    ablation_rows.append({
        "category": category,
        "question": question,
        "rag_retrieval_seconds": float(retrieval_match.group(1)) if retrieval_match else None,
        "rag_generation_seconds": float(generation_match.group(1)) if generation_match else None,
        "rag_total_seconds": round(rag_seconds, 3),
        "baseline_seconds": round(baseline_seconds, 3),
        "rag_citations": len(CITATION_PATTERN.findall(rag_body)),
        "baseline_citations": len(CITATION_PATTERN.findall(baseline_answer)),
        "rag_abstained": NOT_FOUND in rag_body.lower(),
        "baseline_abstained": NOT_FOUND in baseline_answer.lower(),
        "rag_answer": rag_answer,
        "baseline_answer": baseline_answer,
        # Manual scores (0-2). Fill these in after checking each claim against the cited evidence.
        "rag_correctness_0_to_2": None,
        "rag_grounding_0_to_2": None,
        "rag_relevance_0_to_2": None,
        "baseline_correctness_0_to_2": None,
        "baseline_grounding_0_to_2": None,
        "baseline_relevance_0_to_2": None,
        "review_notes": "",
    })
    print(f"[{category}] RAG {rag_seconds:.2f}s · baseline {baseline_seconds:.2f}s")

ablation_results = pd.DataFrame(ablation_rows)
summary_columns = [
    "category", "rag_retrieval_seconds", "rag_generation_seconds", "rag_total_seconds",
    "baseline_seconds", "rag_citations", "baseline_citations", "rag_abstained", "baseline_abstained",
]
display(ablation_results[summary_columns])
print("\nMean latency (s):", ablation_results[["rag_total_seconds", "baseline_seconds"]].mean().round(2).to_dict())
ablation_results.to_csv("cti_rag_vs_baseline_evaluation.csv", index=False)
print("Saved cti_rag_vs_baseline_evaluation.csv. Fill in the manual 0-2 scores before reporting results.")


# Serve the RAG Pipeline for the Vercel Website

The website on Vercel is only the user interface; Gemma 2 2B needs the Colab GPU. This cell:

1. Starts a FastAPI server inside this Colab runtime that calls the same `process_soc_query()` used by the Gradio app (`GET /api/health`, `POST /api/query`).
2. Opens a temporary Cloudflare quick tunnel and prints a public `https://….trycloudflare.com` URL.
3. Prints an access key (set `CTI_ACCESS_KEY` in Colab Secrets to keep it fixed). Requests without the key are rejected, so other people cannot use your GPU session.

Paste the URL and key into the website's connection panel. The URL changes every time this cell runs and stops working when the Colab runtime disconnects.


In [ ]:
# ---------------------------------------------------------------------
# Serve the RAG pipeline as a JSON API for the Vercel website
# ---------------------------------------------------------------------
# Run this cell AFTER the Gradio cell. It reuses process_soc_query() so the
# website returns exactly the same grounded answers as the Gradio interface.
# A Cloudflare quick tunnel gives a temporary public https URL; paste it into
# the "Backend URL" box on the Vercel site. The URL changes on every run.
import os
import re
import secrets
import shutil
import subprocess
import threading
import time
from datetime import datetime, timezone

import uvicorn
from fastapi import FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

API_PORT = int(os.getenv("CTI_API_PORT", "8000"))


def _read_secret(name):
    value = os.getenv(name)
    if value:
        return value.strip()
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value.strip() if value else None
    except Exception:
        return None


# Optional shared key. If CTI_ACCESS_KEY is unset, a random key is generated and printed
# below; enter it in the website's "Access key" box so strangers cannot use your GPU.
CTI_ACCESS_KEY = _read_secret("CTI_ACCESS_KEY") or secrets.token_urlsafe(12)

_gpu_lock = threading.Lock()
_TIMING_PATTERNS = {
    "retrieval_seconds": re.compile(r"retrieval\s+([\d.]+)s"),
    "generation_seconds": re.compile(r"Gemma generation\s+([\d.]+)s"),
    "total_seconds": re.compile(r"total\s+([\d.]+)s"),
}


def _parse_timings(answer):
    found = {}
    for name, pattern in _TIMING_PATTERNS.items():
        match = pattern.search(answer or "")
        found[name] = float(match.group(1)) if match else None
    return found


class QueryRequest(BaseModel):
    question: str = Field(..., min_length=1, max_length=8000)
    mode: str = Field("cti", pattern="^(cti|email)$")
    use_live_api: bool = False


def _no_progress(*args, **kwargs):
    return None


api = FastAPI(title="CTI RAG Assistant API", version="2.0")
api.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["Content-Type", "X-Access-Key"],
)


def _check_key(x_access_key):
    if not secrets.compare_digest(str(x_access_key or ""), CTI_ACCESS_KEY):
        raise HTTPException(status_code=401, detail="Invalid or missing access key.")


@api.get("/api/health")
def health(x_access_key: str | None = Header(default=None)):
    _check_key(x_access_key)
    return {
        "status": "ok",
        "model": globals().get("model_id", "google/gemma-2-2b-it"),
        "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
        "top_k": globals().get("TOP_K"),
        "candidate_k": globals().get("CANDIDATE_K"),
        "max_new_tokens": globals().get("MAX_NEW_TOKENS"),
        "vector_chunks": len(globals().get("docs", []) or []),
        "cve_records": int(len(cve_df)) if "cve_df" in globals() else None,
        "ransomware_claims": int(len(ransomware_df)) if "ransomware_df" in globals() else None,
        "phish_domains": len(globals().get("IOC_DOMAINS", set()) or set()),
        "phish_feed_status": globals().get("IOC_FEED_STATUS"),
        "phish_feed_updated": globals().get("IOC_FEED_LAST_UPDATED"),
        "live_api_configured": bool(get_threatcluster_api_key()) if "get_threatcluster_api_key" in globals() else False,
        "server_time_utc": datetime.now(timezone.utc).isoformat(),
    }


@api.post("/api/query")
def query(body: QueryRequest, x_access_key: str | None = Header(default=None)):
    _check_key(x_access_key)
    analysis_mode = "Email triage" if body.mode == "email" else "CTI research"
    started = time.perf_counter()
    with _gpu_lock:  # One request at a time, like the Gradio queue.
        answer, sources = process_soc_query(
            body.question, analysis_mode, use_live_api=body.use_live_api, progress=_no_progress
        )
    return {
        "mode": body.mode,
        "answer_markdown": answer,
        "sources_markdown": sources,
        "timings": {**_parse_timings(answer), "server_seconds": round(time.perf_counter() - started, 3)},
    }


def _start_api_server():
    config = uvicorn.Config(api, host="0.0.0.0", port=API_PORT, log_level="warning")
    server = uvicorn.Server(config)
    threading.Thread(target=server.run, daemon=True).start()
    for _ in range(50):
        if server.started:
            return server
        time.sleep(0.2)
    raise RuntimeError("The API server did not start.")


def _start_cloudflare_tunnel():
    binary = shutil.which("cloudflared") or "/usr/local/bin/cloudflared"
    if not os.path.exists(binary):
        subprocess.run(
            ["wget", "-q", "-O", binary,
             "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
            check=True,
        )
        os.chmod(binary, 0o755)
    process = subprocess.Popen(
        [binary, "tunnel", "--no-autoupdate", "--url", f"http://localhost:{API_PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    deadline = time.time() + 60
    for line in process.stdout:
        found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if found:
            threading.Thread(target=lambda: [None for _ in process.stdout], daemon=True).start()
            return process, found.group(0)
        if time.time() > deadline:
            break
    raise RuntimeError("Cloudflare tunnel URL was not received. Re-run this cell.")


if "api_server" not in globals() or not getattr(api_server, "started", False):
    api_server = _start_api_server()
if "tunnel_process" in globals() and tunnel_process.poll() is None:
    tunnel_process.terminate()
tunnel_process, BACKEND_URL = _start_cloudflare_tunnel()

print("Backend API is running.")
print("Backend URL :", BACKEND_URL)
print("Access key  :", CTI_ACCESS_KEY)
print("Paste both into the website's settings, or open:")
print(f"  https://<your-vercel-site>/?backend={BACKEND_URL}")
print("Keep this Colab tab open; the URL stops working when the runtime disconnects.")
